# ENEM 2020–2024: distribuição das notas por UF da escola


André Marcelno Watanabe - 14558311  
Camila Aya Saito - 15635649  
Guilherme Vaz de Oliveira Taratá - 10817476  
Vinicius Moraes de Carvalho - 15642432



# Etapa 1 - Análise das notas do ENEM

**Pergunta desta etapa:** como as notas dos participantes com UF da escola informada se distribuem e variam entre estados, regiões, áreas e anos?

A pergunta maior do projeto é a relação entre condições socioeconômicas, condições escolares e desempenho. Aqui preparamos a base e descrevemos os padrões

**Base de dados utilizada nessa fase:** Microdados do ENEM (INEP)

### O que significa uma linha?

No arquivo bruto de resultados, cada linha corresponde a uma inscrição naquela edição, com as notas em colunas. Na amostra reorganizada, cada linha é uma nota de uma área; os sorteios são feitos separadamente por área. Já nas tabelas territoriais, cada linha é UF × ano × área. Uma inscrição pode ter notas ausentes. As escalas das provas diferem, por isso os painéis fixam área e ano, e as amostras por área não são unidas como se contivessem as mesmas pessoas.

### Critérios da análise

- Território: `SG_UF_ESC`, a UF da escola informada (escola que o aluno estuda); não se preenche uma ausência usando UF da prova ou residência.
- Seleção: elegibilidade própria de cada área, seguida de 10% de cada ano × UF da escola × área, sem reposição.
- Medida principal: **mediana em todos os recortes**. Média e dispersão continuam disponíveis.
- Shapiro–Wilk: diagnóstico exploratório de normalidade; seu p-valor não escolhe a medida do mapa.
- Participação: contagens anteriores ao sorteio, sem multiplicar a amostra por dez.
- Rankings: ordem descritiva da amostra; diferenças pequenas não são automaticamente significativas.

Não há uma “nota verdadeira do estado” descoberta por um teste. O que estimamos aqui é um centro da distribuição dos participantes incluídos no recorte. A cobertura da UF escolar e a composição dos participantes limitam a generalização.



## 1. Conectar o Drive e escolher o modo de execução

Conecte o Drive e indique a pasta dos arquivos. O caminho abaixo usa `Meu Drive/ENEM`.

- `PROCESSAR_MICRODADOS = False`: carrega os CSVs já prontos
- `PROCESSAR_MICRODADOS = True`: refaz a amostra a partir dos arquivos originais. É a etapa demorada
- `EXPORTAR_TABELAS = True`: salva as tabelas de resumo no final

Mudar os filtros dos gráficos não exige refazer a amostra nem ativar a exportação


A configuração seguinte também funciona fora do Colab. A etapa 1 mantém seus próprios arquivos de amostra e resumo. A etapa 2 pode ser executada separadamente com a pasta `estudo_enem_revisado`, sem reler os microdados. Não é preciso sortear outra amostra para executar os novos testes.

In [ ]:
import gc
import json
from pathlib import Path
from html import escape

import ipywidgets as widgets
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import HTML, IFrame, Markdown, display
import plotly.graph_objects as go
import plotly.io as pio
from scipy.stats import probplot, shapiro

SEMENTE = 42
FRACAO_AMOSTRA = 0.10
TAMANHO_PARTE = 250_000
ANOS = [2020, 2021, 2022, 2023, 2024]

#ordem usada nos filtros e nos gráficos.
AREAS = [
    "Linguagens",
    "Ciências Humanas",
    "Ciências da Natureza",
    "Matemática",
    "Redação",
]


#o mesmo fundo nos graficos
px.defaults.template = "plotly_white"
pio.templates.default = "plotly_white"
px.defaults.color_discrete_sequence = ["#167D9A", "#D97732", "#548744", "#9871B8"]
pio.templates["plotly_white"].layout.update(
    font=dict(family="Arial, sans-serif", size=14, color="#243447"),
    title=dict(font=dict(size=20), x=0.02, xanchor="left"),
    paper_bgcolor="white",
    plot_bgcolor="white",
    hoverlabel=dict(font_size=14),
    separators=",.",
)
REGIOES = ["Norte","Nordeste","Centro-Oeste","Sudeste", "Sul"]
cores_regioes = dict(
    zip(REGIOES, ["#0072B2","#E69F00", "#009E73", "#D55E00", "#CC79A7"])
)


In [ ]:
try:
    from google.colab import drive, output
except ImportError:
    PASTA_DADOS = Path.cwd()
    # Ao abrir o notebook dentro do pacote, os microdados ficam três pastas acima.
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / 'MICRODADOS_ENEM_2020.csv').exists():
            PASTA_DADOS = pasta
            break
else:
    drive.mount('/content/drive')
    output.enable_custom_widget_manager()
    PASTA_DADOS = Path('/content/drive/MyDrive/ENEM')

# False usa os resumos existentes. True relê os microdados e refaz a amostra.
PROCESSAR_MICRODADOS = False
EXPORTAR_TABELAS = False
print('Pasta da etapa 1:', PASTA_DADOS)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Pasta da etapa 1: /content/drive/MyDrive/ENEM


In [ ]:
#relaciono cada ano ao arquivo que contem as notas e a UF da escola
arquivos_por_ano = {
    2020: PASTA_DADOS / "MICRODADOS_ENEM_2020.csv",
    2021: PASTA_DADOS / "MICRODADOS_ENEM_2021.csv",
    2022: PASTA_DADOS / "MICRODADOS_ENEM_2022.csv",
    2023: PASTA_DADOS / "MICRODADOS_ENEM_2023.csv",
    2024: PASTA_DADOS / "RESULTADOS_2024.csv",
}

#arquivos criados e utilizados aqui
ARQUIVO_AMOSTRA = PASTA_DADOS / "enem_amostra_uf_escola_2020_2024.csv"
ARQUIVO_VALIDACAO = PASTA_DADOS / "enem_validacao_amostra_uf_escola_2020_2024.csv"
ARQUIVO_COBERTURA = PASTA_DADOS / "enem_cobertura_uf_escola_2020_2024.csv"
ARQUIVO_MAPA = PASTA_DADOS / "malha_ufs_plotly.geojson"

conferencia = []

for ano,caminho in arquivos_por_ano.items():
    conferencia.append(
        {
            "ANO": ano,
            "ARQUIVO": caminho.name,
            "ENCONTRADO": caminho.exists(),
        }
    )

if PROCESSAR_MICRODADOS:
    display(pd.DataFrame(conferencia))
print("amostra pronta:", ARQUIVO_AMOSTRA.exists())
print("mapa encontrado:", ARQUIVO_MAPA.exists())


#
if PROCESSAR_MICRODADOS:
    entradas = list(arquivos_por_ano.values())
else:
    entradas = [ARQUIVO_AMOSTRA, ARQUIVO_VALIDACAO, ARQUIVO_COBERTURA]

faltantes = [caminho.name for caminho in entradas if not caminho.exists()]
if faltantes:
    raise FileNotFoundError("coloque na pasta ENEM: " + ", ".join(faltantes))


amostra pronta: True
mapa encontrado: True


## 2. Selecionar as colunas e organizar as UFs

Usamos nota, presença e situação da redação para selecionar os registros. A UF da escola define o território e o ano vem do arquivo. Região, nome e código do estado são obtidos a partir da UF.

O primeiro dicionário liga cada área às colunas dos microdados. Redação usa a presença de Linguagens, pois as provas acontecem no mesmo dia. Renda, rede escolar e escolaridade familiar ficam para uma próxima extração, depois de conferir a disponibilidade em cada ano.


In [ ]:
#colunas  de nota e presença de cada área
colunas_por_area = {
    "Linguagens": ("NU_NOTA_LC", "TP_PRESENCA_LC"),
    "Ciências Humanas": ("NU_NOTA_CH", "TP_PRESENCA_CH"),
    "Ciências da Natureza": ("NU_NOTA_CN", "TP_PRESENCA_CN"),
    "Matemática": ("NU_NOTA_MT", "TP_PRESENCA_MT"),
    "Redação": ("NU_NOTA_REDACAO", "TP_PRESENCA_LC"),
}

#regiao de cada UF
regiao_por_uf = {
    "AC": "Norte",
    "AP": "Norte",
    "AM": "Norte",
    "PA": "Norte",
    "RO": "Norte",
    "RR": "Norte",
    "TO": "Norte",
    "AL": "Nordeste",
    "BA": "Nordeste",
    "CE": "Nordeste",
    "MA": "Nordeste",
    "PB": "Nordeste",
    "PE": "Nordeste",
    "PI": "Nordeste",
    "RN": "Nordeste",
    "SE": "Nordeste",
    "DF": "Centro-Oeste",
    "GO": "Centro-Oeste",
    "MT": "Centro-Oeste",
    "MS": "Centro-Oeste",
    "ES": "Sudeste",
    "MG": "Sudeste",
    "RJ": "Sudeste",
    "SP": "Sudeste",
    "PR": "Sul",
    "RS": "Sul",
    "SC": "Sul",
}

#codigo do IBGE usado no mapa
codigo_por_uf = {
    "RO": "11",
    "AC": "12",
    "AM": "13",
    "RR": "14",
    "PA": "15",
    "AP": "16",
    "TO": "17",
    "MA": "21",
    "PI": "22",
    "CE": "23",
    "RN": "24",
    "PB": "25",
    "PE": "26",
    "AL": "27",
    "SE": "28",
    "BA": "29",
    "MG": "31",
    "ES": "32",
    "RJ": "33",
    "SP": "35",
    "PR": "41",
    "SC": "42",
    "RS": "43",
    "MS": "50",
    "MT": "51",
    "GO": "52",
    "DF": "53",
}

#nome mostrados nos graficos
nome_por_uf = {
    "AC": "Acre",
    "AL": "Alagoas",
    "AP": "Amapá",
    "AM": "Amazonas",
    "BA": "Bahia",
    "CE": "Ceará",
    "DF": "Distrito Federal",
    "ES": "Espírito Santo",
    "GO": "Goiás",
    "MA": "Maranhão",
    "MT": "Mato Grosso",
    "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais",
    "PA": "Pará",
    "PB": "Paraíba",
    "PR": "Paraná",
    "PE": "Pernambuco",
    "PI": "Piauí",
    "RJ": "Rio de Janeiro",
    "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul",
    "RO": "Rondônia",
    "RR": "Roraima",
    "SC": "Santa Catarina",
    "SP": "São Paulo",
    "SE": "Sergipe",
    "TO": "Tocantins",
}

ufs_validas = set(regiao_por_uf)


## 3. Preparar a leitura dos microdados

Lemos apenas as colunas necessárias e mantemos seus nomes oficiais para facilitar a consulta ao dicionário do INEP. As siglas são texto, os códigos são inteiros e as notas são números decimais.

A leitura é feita em blocos, mas reunimos os blocos de um ano antes do sorteio. Por isso, ainda é preciso memória para um ano dessas colunas. Os cinco anos não ficam carregados juntos nesta etapa.


In [ ]:
#colunas para criar a nova amostra
colunas_necessarias = [
    "SG_UF_ESC",
    "TP_PRESENCA_CN",
    "TP_PRESENCA_CH",
    "TP_PRESENCA_LC",
    "TP_PRESENCA_MT",
    "NU_NOTA_CN",
    "NU_NOTA_CH",
    "NU_NOTA_LC",
    "NU_NOTA_MT",
    "TP_STATUS_REDACAO",
    "NU_NOTA_REDACAO",
]

#tipos menores diminuem o uso de memória durante a leitura
tipos_microdados = {
    "SG_UF_ESC": "string",
    "TP_PRESENCA_CN": "Int8",
    "TP_PRESENCA_CH": "Int8",
    "TP_PRESENCA_LC": "Int8",
    "TP_PRESENCA_MT": "Int8",
    "TP_STATUS_REDACAO": "Int8",
    "NU_NOTA_CN": "Float64",
    "NU_NOTA_CH": "Float64",
    "NU_NOTA_LC": "Float64",
    "NU_NOTA_MT": "Float64",
    "NU_NOTA_REDACAO": "Float64",
}


## 4. Selecionar os registros e sortear a amostra

Esta célula só processa os microdados se `PROCESSAR_MICRODADOS = True`. Para cada ano e área:

1. Mantemos presença igual a 1 e nota preenchida. Em Redação, exigimos também `TP_STATUS_REDACAO = 1`; outras situações ficam de fora, mesmo quando têm nota.
2. Contamos os elegíveis com e sem UF da escola reconhecida.
3. Entre os que têm UF reconhecida, sorteamos 10% de cada UF, sem reposição. O tamanho é arredondado para um número inteiro: 2.000 elegíveis geram 200 notas, por exemplo. O sorteio é feito uma única vez por grupo.
4. Salvamos a amostra e as contagens usadas na validação e nos gráficos de participação e cobertura.

Notas iguais são mantidas, pois podem pertencer a pessoas diferentes. A amostra não guarda identificador individual e não permite uma futura junção pessoa a pessoa. Os CSVs anteriores só são substituídos depois que todos os anos terminam de ser processados.


In [ ]:
if PROCESSAR_MICRODADOS:
    #confere as colunas de todos os arquivos antes de começar
    for caminho in arquivos_por_ano.values():
        cabecalho = pd.read_csv(caminho, sep=";", encoding="latin1", nrows=0)
        ausentes = set(colunas_necessarias) - set(cabecalho.columns)
        if ausentes:
            raise ValueError(f"Colunas ausentes em {caminho.name}: {sorted(ausentes)}")

    #grava em arquivo temporário ate terminar todos os anos
    amostra_em_preparo = ARQUIVO_AMOSTRA.with_suffix(".em_preparo.csv")

    escrever_cabecalho = True
    linhas_validacao = []
    linhas_cobertura = []

    #le um ano por vez para poupar memoria
    for ano in ANOS:
        caminho = arquivos_por_ano[ano]

        if not caminho.exists():
            raise FileNotFoundError(f"Arquivo não encontrado: {caminho.name}")

        print(f"Lendo {ano}: {caminho.name}")
        partes = []

        #le apenas as colunas necessárias, em blocos
        leitor = pd.read_csv(
            caminho,
            sep=";",
            encoding="latin1",
            usecols=colunas_necessarias,
            dtype=tipos_microdados,
            chunksize=TAMANHO_PARTE,
            low_memory=False,
        )

        for numero_parte, parte in enumerate(leitor, start=1):
            #remove espaços e deixa as siglas em maiúsculas
            parte["SG_UF_ESC"] = parte["SG_UF_ESC"].str.strip().str.upper()
            partes.append(parte)

            if numero_parte % 5 == 0:
                print(f"  {numero_parte * TAMANHO_PARTE:,} participantes lidos")

        dados_ano = pd.concat(partes, ignore_index=True)
        del partes, parte
        gc.collect()

        #so aceita siglas presentes na lista de UFs
        uf_escola_valida = dados_ano["SG_UF_ESC"].isin(ufs_validas)

        #a selecao e o sorteio são feitos separadamente para cada área
        for area, (coluna_nota, coluna_presenca) in colunas_por_area.items():
            nota_valida = (
                dados_ano[coluna_presenca].eq(1) & dados_ano[coluna_nota].notna()
            )

            #em Redação, mantemos apenas a situação de correção 1
            if area == "Redação":
                nota_valida &= dados_ano["TP_STATUS_REDACAO"].eq(1)

            total_notas_validas = int(nota_valida.sum())
            total_com_uf_escola = int((nota_valida & uf_escola_valida).sum())

            #conta também quem não informou a UF da escola, antes do sorteio
            linhas_cobertura.append(
                {
                    "ANO": ano,
                    "AREA": area,
                    "NOTAS_VALIDAS": total_notas_validas,
                    "NOTAS_COM_UF_ESCOLA": total_com_uf_escola,
                    "NOTAS_SEM_UF_ESCOLA": total_notas_validas - total_com_uf_escola,
                    "COBERTURA_UF_ESCOLA": total_com_uf_escola
                    / total_notas_validas
                    * 100,
                }
            )

            elegiveis = dados_ano.loc[
                nota_valida & uf_escola_valida,
                ["SG_UF_ESC", coluna_nota],
            ].copy()

            contagem_elegiveis = elegiveis["SG_UF_ESC"].value_counts()

            #sorteia 10% separadamente dentro de cada UF da escola
            amostra_area = (
                elegiveis.groupby("SG_UF_ESC", group_keys=False)
                .sample(frac=FRACAO_AMOSTRA, replace=False, random_state=SEMENTE)
                .reset_index(drop=True)
            )

            contagem_amostra = amostra_area["SG_UF_ESC"].value_counts()

            #cada linha da amostra guarda uma nota de uma área
            amostra_area = amostra_area.rename(
                columns={
                    "SG_UF_ESC": "UF_ESCOLA",
                    coluna_nota: "NOTA",
                }
            )
            amostra_area.insert(0, "ANO", ano)
            amostra_area["NOME_UF"] = amostra_area["UF_ESCOLA"].map(nome_por_uf)
            amostra_area["COD_UF_IBGE"] = amostra_area["UF_ESCOLA"].map(codigo_por_uf)
            amostra_area["REGIAO"] = amostra_area["UF_ESCOLA"].map(regiao_por_uf)
            amostra_area["AREA"] = area
            amostra_area["TOTAL_ELEGIVEIS_ESTRATO"] = amostra_area["UF_ESCOLA"].map(
                contagem_elegiveis
            )
            amostra_area["TAMANHO_AMOSTRA_ESTRATO"] = amostra_area["UF_ESCOLA"].map(
                contagem_amostra
            )
            amostra_area["PERCENTUAL_AMOSTRADO"] = (
                amostra_area["TAMANHO_AMOSTRA_ESTRATO"]
                / amostra_area["TOTAL_ELEGIVEIS_ESTRATO"]
                * 100
            )

            #acrescenta a amostra da área ao CSV e libera a memória em seguida
            amostra_area.to_csv(
                amostra_em_preparo,
                sep=";",
                encoding="utf-8-sig" if escrever_cabecalho else "utf-8",
                index=False,
                mode="w" if escrever_cabecalho else "a",
                header=escrever_cabecalho,
            )
            escrever_cabecalho = False

            #guarda as contagens de cada grupo de ano, UF e área.
            for uf, total in contagem_elegiveis.items():
                quantidade = int(contagem_amostra.get(uf, 0))
                linhas_validacao.append(
                    {
                        "ANO": ano,
                        "UF_ESCOLA": uf,
                        "REGIAO": regiao_por_uf[uf],
                        "AREA": area,
                        "REGISTROS_ELEGIVEIS": int(total),
                        "AMOSTRA": quantidade,
                        "PERCENTUAL": quantidade / int(total) * 100,
                    }
                )

            print(f"  {area}: {len(amostra_area):,} notas sorteadas")

            del elegiveis, amostra_area
            gc.collect()

        del dados_ano
        gc.collect()

    #salva as contagens em arquivos temporários
    pd.DataFrame(linhas_validacao).to_csv(
        ARQUIVO_VALIDACAO.with_suffix(".em_preparo.csv"),
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    pd.DataFrame(linhas_cobertura).to_csv(
        ARQUIVO_COBERTURA.with_suffix(".em_preparo.csv"),
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )

    #substitui os CSVs anteriores depois de concluir o processamento
    amostra_em_preparo.replace(ARQUIVO_AMOSTRA)
    ARQUIVO_VALIDACAO.with_suffix(".em_preparo.csv").replace(ARQUIVO_VALIDACAO)
    ARQUIVO_COBERTURA.with_suffix(".em_preparo.csv").replace(ARQUIVO_COBERTURA)
    print("Nova amostra concluída.")
else:
    print("A amostra pronta será carregada. Nenhum microdado será reprocessado.")

A amostra pronta será carregada. Nenhum microdado será reprocessado.


## 5. Carregar a amostra e conferir seu tamanho

Carregamos os CSVs de amostra, validação e cobertura. A diferença entre o tamanho sorteado e 10% dos elegíveis deve ser de até meia linha, por causa do arredondamento. Essa conferência verifica o tamanho; não prova a aleatoriedade nem elimina o viés de participação no ENEM.

Lemos as notas em precisão dupla e arredondamos para a casa decimal original, removendo resíduos de arquivos antigos. As medidas calculadas continuam com sua precisão: a mediana entre 535,2 e 535,3, por exemplo, é 535,25.


In [ ]:
#category e int ajudam a economizar memoria
tipos_amostra = {
    "ANO": "int16",
    "UF_ESCOLA": "category",
    "NOME_UF": "category",
    "COD_UF_IBGE": "string",
    "REGIAO": "category",
    "AREA": "category",
    "NOTA": "float64",
    "TOTAL_ELEGIVEIS_ESTRATO": "int32",
    "TAMANHO_AMOSTRA_ESTRATO": "int32",
    "PERCENTUAL_AMOSTRADO": "float32",
}

#carrega a amostra, a validação e a cobertura da UF escolar
dados_analise = pd.read_csv(
    ARQUIVO_AMOSTRA,
    sep=";",
    encoding="utf-8-sig",
    dtype=tipos_amostra,
)

validacao_amostra = pd.read_csv(
    ARQUIVO_VALIDACAO,
    sep=";",
    encoding="utf-8-sig",
)

cobertura_uf_escola = pd.read_csv(
    ARQUIVO_COBERTURA,
    sep=";",
    encoding="utf-8-sig",
)

#aceita a diferença de arredondamento no tamanho sorteado
validacao_amostra["DIFERENCA_DO_ESPERADO"] = (
    validacao_amostra["AMOSTRA"]
    - validacao_amostra["REGISTROS_ELEGIVEIS"] * FRACAO_AMOSTRA
).abs()

amostra_correta = validacao_amostra["DIFERENCA_DO_ESPERADO"].le(0.51).all()

print("Linhas da nova amostra:", f"{len(dados_analise):,}")
print("Estratos de ano × UF da escola × área:", len(validacao_amostra))
print("Amostragem de 10% correta, considerando o arredondamento:", amostra_correta)


# Remove resíduos decimais de CSVs antigos, gravados com float32.
dados_analise["NOTA"] = dados_analise["NOTA"].round(1)


Linhas da nova amostra: 1,910,983
Estratos de ano × UF da escola × área: 675
Amostragem de 10% correta, considerando o arredondamento: True


## 6. Conferir se as três tabelas correspondem

Antes dos gráficos, verificamos os anos, áreas, UFs, regiões, notas e valores ausentes. Também conferimos se:

- a contagem da amostra bate com a validação em cada ano, UF e área;
- a soma dos elegíveis por UF bate com a cobertura;
- os elegíveis com UF e sem UF somam o total elegível.

Se uma conferência falhar, a execução para, evitando misturar CSVs de versões diferentes. Não removemos linhas por terem a mesma nota: sem identificador, não sabemos se são registros duplicados ou pessoas diferentes. As contagens por área pressupõem uma linha por participante no arquivo de origem.


In [ ]:
chaves = ["ANO", "UF_ESCOLA", "AREA"]
assert not dados_analise.empty, "A amostra está vazia."
assert (
    dados_analise[chaves + ["REGIAO", "NOTA"]].notna().all().all()
), "Há ausentes na seleção."
assert dados_analise["UF_ESCOLA"].isin(ufs_validas).all(), "Há UF desconhecida."
assert dados_analise["ANO"].isin(ANOS).all(), "Há ano fora do período. "
assert dados_analise["AREA"].isin(AREAS).all(), "Há área desconhecida."
assert np.isfinite(dados_analise["NOTA"]).all(), "Há nota com valor não finito."
assert dados_analise["NOTA"].ge(0).all(), "Há nota negativa."
assert not validacao_amostra.duplicated(chaves).any(), "Estrato repetido na validação."
assert not cobertura_uf_escola.duplicated(["ANO", "AREA"]).any()
assert amostra_correta, "O tamanho não corresponde aos 10% definidos."
regioes_esperadas = dados_analise["UF_ESCOLA"].astype(str).map(regiao_por_uf)
assert (regioes_esperadas == dados_analise["REGIAO"].astype(str)).all()

#confere se o número de notas lidas bate com a tabela de validação
contagem_lida = (
    dados_analise.groupby(chaves, observed=True).size().rename("LIDAS").reset_index()
)
conferencia = validacao_amostra.merge(
    contagem_lida, on=chaves, how="outer", validate="one_to_one"
)
assert conferencia[["AMOSTRA", "LIDAS"]].notna().all().all()
assert (
    conferencia["AMOSTRA"].eq(conferencia["LIDAS"]).all()
), "Amostra e validação não correspondem."
assert validacao_amostra["REGISTROS_ELEGIVEIS"].ge(validacao_amostra["AMOSTRA"]).all()

#confere se a soma das UFs bate com a cobertura de cada ano e área
totais = (
    validacao_amostra.groupby(["ANO", "AREA"])["REGISTROS_ELEGIVEIS"]
    .sum()
    .reset_index()
)
totais = totais.merge(
    cobertura_uf_escola, on=["ANO", "AREA"], how="outer", validate="one_to_one"
)
assert totais.notna().all().all()
assert totais["REGISTROS_ELEGIVEIS"].eq(totais["NOTAS_COM_UF_ESCOLA"]).all()
total_conferido = totais["NOTAS_COM_UF_ESCOLA"] + totais["NOTAS_SEM_UF_ESCOLA"]
assert totais["NOTAS_VALIDAS"].eq(total_conferido).all()

cobertura_calculada = 100 * totais["NOTAS_COM_UF_ESCOLA"] / totais["NOTAS_VALIDAS"]
assert np.allclose(totais["COBERTURA_UF_ESCOLA"], cobertura_calculada)

print(
    "aprovada:", len(dados_analise), "notas;", len(conferencia), "estratos."
)
print(
    "A amostra descreve elegíveis com UF da escola informada; não todos os inscritos."
)


aprovada: 1910983 notas; 675 estratos.
A amostra descreve elegíveis com UF da escola informada; não todos os inscritos.


### Como os painéis são exibidos

As três funções abaixo cuidam apenas da apresentação: exibem o gráfico, formatam as tabelas e organizam os filtros. Os cálculos continuam nas células de cada análise. Cada gráfico usa uma área própria de HTML para evitar conflitos ao atualizar as saídas no Colab.


In [ ]:
def exibir_grafico(grafico):
    """Exibe o gráfico"""
    altura = grafico.layout.height or 500
    grafico.update_layout(autosize=True, width=None, height=altura)
    pagina = grafico.to_html(
        include_plotlyjs="cdn",
        full_html=True,
        config={
            "responsive": True,
            "displaylogo": False,
            "scrollZoom": False,
            "modeBarButtonsToRemove": ["select2d", "lasso2d"],
            "toImageButtonOptions": {"format": "png", "scale": 2},
        },
    )
    pagina = pagina.replace("<head>", "<head><style>body{margin:0}</style>")
    quadro = IFrame(
        src="about:blank",
        width="100%",
        height=altura + 4,
        extras=[
            f'srcdoc="{escape(pagina)}"',
            'title="Gráfico interativo"',
            'style="min-width:600px;border:0;display:block;background:white;border-radius:8px;"',
        ],
    )
    display(
        HTML(
            '<div style="width:100%;overflow-x:auto;">'
            + quadro._repr_html_()
            + "</div>"
        )
    )


def exibir_tabela(tabela):
    """Formata os detalhes sem ocupar uma saída inteira com linhas e colunas."""
    nomes = {
        "LOCALIZACAO": "Localização",
        "REDE": "Rede",
        "RECURSO": "Recurso",
        "PERCENTUAL": "Percentual (%)",
        "ESCOLAS": "Escolas no recorte",
        "INFORMADAS": "Escolas com resposta",
        "COM_RECURSO": "Escolas com recurso",
        "N_URBANA": "Notas urbanas",
        "N_RURAL": "Notas rurais",
        "MEDIANA_URBANA": "Mediana urbana",
        "MEDIANA_RURAL": "Mediana rural",
        "DIFERENCA": "Rural − urbana (pontos)",
        "PCT_RURAL": "Elegíveis em escolas rurais (%)",
        "PCT_PRIVADA": "Elegíveis na rede privada (%)",
        "NOTA_ALVO": "Mediana",
        "N_AMOSTRA": "Notas na amostra",
        "ANO": "Ano",
        "AREA": "Área",
        "UF_ESCOLA": "UF da escola",
        "NOME_UF": "Estado",
        "REGIAO": "Região",
        "N": "Notas na amostra",
        "REGISTROS_ELEGIVEIS": "Elegíveis",
        "AMOSTRA": "Amostrados",
        "MEDIA": "Média",
        "MEDIANA": "Mediana",
        "DESVIO_PADRAO": "Desvio-padrão",
        "ASSIMETRIA": "Assimetria",
        "MINIMO": "Mínimo",
        "MAXIMO": "Máximo",
    }
    formatada = (
        tabela.rename(columns=nomes)
        .style.hide(axis="index")
        .format(precision=2, decimal=",", thousands=".")
        .set_table_styles(
            [
                {
                    "selector": "",
                    "props": "border-collapse:collapse;font:14px Arial;color:#243447;background:white;width:100%;",
                },
                {
                    "selector": "th",
                    "props": "padding:12px;text-align:left;background:#edf3f7;white-space:nowrap;",
                },
                {
                    "selector": "td",
                    "props": "padding:10px 12px;border-bottom:1px solid #e2e8f0;white-space:nowrap;",
                },
            ]
        )
    )
    if "p-valor" in tabela.columns:
        formatada = formatada.format(
            {"p-valor": "{:.3g}", "W de Shapiro": "{:.4f}"},
            subset=["p-valor", "W de Shapiro"],
            decimal=",",
        )
    display(
        HTML(
            '<div style="overflow:auto;max-height:360px;margin:12px 0;">'
            + formatada.to_html()
            + "</div>"
        )
    )


def montar_painel(funcao, **controles):
    """Organiza os filtros e mantém apenas a saída do recorte atual."""
    for controle in controles.values():
        controle.layout = widgets.Layout(width="310px", max_width="100%")
        controle.style.description_width = "initial"
    filtros = widgets.Box(
        list(controles.values()),
        layout=widgets.Layout(
            display="flex", flex_flow="row wrap", gap="8px", width="100%"
        ),
    )
    saida = widgets.interactive_output(funcao, controles)
    saida.layout = widgets.Layout(width="100%", height="auto")
    display(widgets.VBox([filtros, saida], layout=widgets.Layout(width="100%")))

## 7. Participantes elegíveis e cobertura da UF escolar

Escolha uma área e alterne **Mostrar** entre as contagens de participantes e a porcentagem com UF da escola informada. Exibimos um gráfico por vez. Marque **Ver tabela do ano** para consultar o ano selecionado.

As contagens vêm de antes do sorteio. Cada participante é contado uma vez na área e no ano. São totais de elegíveis, e a cobertura é a parcela deles com UF conhecida — não é uma taxa de comparecimento. Mudanças na cobertura podem alterar o perfil do grupo comparado.

Nos painéis, cada `def` reúne os passos de um gráfico. `montar_painel` organiza os filtros e atualiza a saída quando uma escolha muda.


In [ ]:
def mostrar_cobertura(area, ano, visualizacao="Participantes", ver_tabela=False):
    tabela = cobertura_uf_escola[cobertura_uf_escola["AREA"] == area].sort_values("ANO")
    if visualizacao == "Participantes":
        barras = tabela.melt(
            id_vars="ANO",
            value_vars=["NOTAS_COM_UF_ESCOLA", "NOTAS_SEM_UF_ESCOLA"],
            var_name="Situação",
            value_name="Participantes elegíveis",
        )
        barras["Situação"] = barras["Situação"].replace(
            {
                "NOTAS_COM_UF_ESCOLA": "Com UF escolar",
                "NOTAS_SEM_UF_ESCOLA": "Sem UF escolar",
            }
        )
        grafico = px.bar(
            barras,
            x="ANO",
            y="Participantes elegíveis",
            color="Situação",
            title=f"Participantes elegíveis por ano<br><sup>{area}</sup>",
            labels={"ANO": "Ano"},
            color_discrete_map={
                "Com UF escolar": "#0072B2",
                "Sem UF escolar": "#B8BEC8",
            },
        )
        grafico.update_layout(
            height=480,
            legend=dict(orientation="h", y=-0.2, title_text=""),
            margin=dict(t=90, b=110, l=80, r=35),
        )
        grafico.update_xaxes(dtick=1)
        exibir_grafico(grafico)
    else:
        linha = px.line(
            tabela,
            x="ANO",
            y="COBERTURA_UF_ESCOLA",
            markers=True,
            title=f"Cobertura da UF da escola<br><sup>{area} · elegíveis com UF informada</sup>",
            labels={
                "ANO": "Ano",
                "COBERTURA_UF_ESCOLA": "Elegíveis com UF escolar (%)",
            },
        )
        linha.update_yaxes(range=[0, 100])
        linha.update_xaxes(dtick=1)
        linha.update_layout(height=460, margin=dict(t=85, b=65, l=75, r=35))
        exibir_grafico(linha)
    detalhe = tabela[tabela["ANO"] == ano].rename(
        columns={
            "NOTAS_VALIDAS": "Elegíveis totais",
            "NOTAS_COM_UF_ESCOLA": "Com UF escolar",
            "NOTAS_SEM_UF_ESCOLA": "Sem UF escolar",
            "COBERTURA_UF_ESCOLA": "Cobertura (%)",
        }
    )
    if ver_tabela:
        exibir_tabela(detalhe.round(2))


montar_painel(
    mostrar_cobertura,
    visualizacao=widgets.Dropdown(
        options=["Participantes", "Cobertura (%)"], description="Mostrar:"
    ),
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela do ano"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Detalhe ano:"),
)


## 8. Onde se concentram os participantes de cada área?

Escolha ano, área, território e região. O gráfico soma os elegíveis ou os amostrados e ordena as barras. O percentual mostra a participação de cada barra no total exibido.

O total de elegíveis vem da contagem completa com UF da escola, sem multiplicar a amostra por dez. Mais participantes não significam maior taxa de participação: para calcular essa taxa, precisaríamos de uma população de referência compatível.


In [ ]:
def mostrar_participantes(ano, area, territorio, regiao, contagem, ver_tabela=False):
    selecao = (validacao_amostra["ANO"] == ano) & (validacao_amostra["AREA"] == area)
    tabela = validacao_amostra[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    coluna = "REGIAO" if territorio == "Região" else "UF_ESCOLA"
    medida = (
        "REGISTROS_ELEGIVEIS" if contagem == "Elegíveis antes do sorteio" else "AMOSTRA"
    )
    grupos = tabela.groupby(coluna, observed=True)
    tabela = grupos[["REGISTROS_ELEGIVEIS", "AMOSTRA"]].sum().reset_index()
    tabela["Participação no recorte (%)"] = 100 * tabela[medida] / tabela[medida].sum()
    tabela = tabela.sort_values(medida)
    grafico = px.bar(
        tabela,
        x=medida,
        y=coluna,
        orientation="h",
        text=medida,
        title=f"{contagem}<br><sup>{area} · {ano} · {regiao}</sup>",
        hover_data={"Participação no recorte (%)": ":.2f"},
        labels={medida: "Participantes na área", coluna: territorio},
    )
    grafico.update_traces(
        texttemplate="%{text:,.0f}", textposition="outside", cliponaxis=False
    )
    grafico.update_layout(
        height=max(420, 27 * len(tabela) + 130), margin=dict(l=115, r=95, t=100, b=65)
    )
    grafico.update_yaxes(title=None)
    grafico.update_xaxes(range=[0, tabela[medida].max() * 1.18])
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(tabela.sort_values(medida, ascending=False).round(2))


montar_painel(
    mostrar_participantes,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    territorio=widgets.Dropdown(
        options=["Região", "UF da escola"], description="Território:"
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    contagem=widgets.Dropdown(
        options=["Elegíveis antes do sorteio", "Amostrados"],
        description="Contagem:",
        layout=widgets.Layout(width="370px"),
    ),
)


## 9. Qual medida representa o desempenho típico?

Definimos previamente o que queremos resumir: **o centro da distribuição das notas no recorte**. Usamos a mediana, ou percentil 50, de forma consistente em todos os estados, anos e áreas. Ela depende da ordenação das notas e é menos sensível a valores muito altos ou muito baixos. Seu uso não exige normalidade.

Exemplo didático: em `[400, 450, 500, 550, 900]`, a média é 560 e a mediana é 500. A nota 900 desloca a média, enquanto a mediana preserva a posição central. Isso não torna a média errada: ela responde à pergunta sobre a soma das notas dividida pela quantidade. Escolhemos a mediana por nosso objetivo de descrever o centro com menor sensibilidade às caudas.

Nenhuma medida resume sozinha toda a distribuição. Se houver dois grupos muito diferentes, até a mediana pode esconder essa divisão. Por isso, sempre a acompanhamos de histograma, quartis, percentis e N. Maior N não corrige automaticamente viés na seleção dos participantes.

### Por que não usar Shapiro para escolher entre média e mediana?

O Shapiro–Wilk testa a hipótese de uma distribuição normal. Ele **não compara a qualidade da média e da mediana**. Um p-valor pequeno não torna a média inválida; um p-valor grande não prova normalidade. Trocar a medida de estado para estado com base no teste ainda mistura alvos diferentes no mesmo mapa.

O diagnóstico fica separado adiante. Não adotamos qui-quadrado para escolher a nota: categorizar notas em faixas e testar frequências não resolve a escolha da medida central.

Referências: [medidas de localização — NIST](https://www.itl.nist.gov/div898/handbook/eda/section3/eda351.htm) e [Shapiro–Wilk — SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html).

### Como ler as medidas

| Medida | Significado e uso |
|---|---|
| N | Quantidade de notas da amostra no recorte; não é o total de inscritos |
| Média | Soma das notas dividida por N |
| Mediana / P50 | Centro da ordenação; com N par, média dos dois valores centrais |
| Q1 / P25 e Q3 / P75 | Limites aproximados dos 50% centrais |
| P5 e P95 | Delimitam aproximadamente os 90% centrais; não são intervalo de confiança |
| Desvio-padrão | Dispersão em torno da média, na unidade de pontos |
| Q3 − Q1 | Dispersão dos 50% centrais; amplitude interquartil |
| Assimetria | Sinal positivo sugere cauda à direita; negativo, à esquerda; zero não prova normalidade |
| Mínimo e máximo | Extremos observados; não significam erro de registro por si só |

Empates, comuns em Redação, podem fazer mais da metade das notas ficar exatamente na mediana. Percentis descrevem a distribuição; eles não medem a incerteza da estimativa.

## 10. Calcular as medidas de cada UF da escola

Criamos uma linha para cada ano, UF da escola e área. `groupby` separa os grupos, `agg` calcula as medidas e `quantile` encontra os percentis. Depois, juntamos os resultados e as contagens pela chave do grupo.

Todas as notas da amostra entram nos cálculos, inclusive as caudas. `NOTA_REPRESENTATIVA` recebe sempre a mediana; o teste de normalidade não altera essa escolha.


In [ ]:
#agrupa as notas por ano, UF da escola e área
colunas_grupo = ["ANO", "UF_ESCOLA", "NOME_UF", "COD_UF_IBGE", "REGIAO", "AREA"]
grupos_uf = dados_analise.groupby(colunas_grupo, observed=True)["NOTA"]

#medidas de posição, dispersão e forma
resumo_uf = grupos_uf.agg(
    MEDIA="mean",
    MEDIANA="median",
    N="count",
    DESVIO_PADRAO="std",
    ASSIMETRIA="skew",
    MINIMO="min",
    MAXIMO="max",
).reset_index()

#percentis usados para observar o centro e as caudas
percentis_uf = (
    grupos_uf.quantile([0.05, 0.25, 0.75, 0.95])
    .unstack()
    .rename(columns={0.05: "P5", 0.25: "Q1", 0.75: "Q3", 0.95: "P95"})
    .reset_index()
)

resumo_uf = resumo_uf.merge(percentis_uf, on=colunas_grupo)
resumo_uf["AMPLITUDE_INTERQUARTIL"] = resumo_uf["Q3"] - resumo_uf["Q1"]

#a nota representativa é sempre a mediana
resumo_uf["NOTA_REPRESENTATIVA"] = resumo_uf["MEDIANA"]


#junta o tamanho da amostra e o total de elegíveis de cada grupo
resumo_uf = resumo_uf.merge(
    validacao_amostra[chaves + ["REGISTROS_ELEGIVEIS"]],
    on=chaves,
    validate="one_to_one",
)
assert resumo_uf["N"].sum() == len(dados_analise)
print(f"Resumo estadual: {len(resumo_uf)} grupos de ano, UF e área.")


Resumo estadual: 675 grupos de ano, UF e área.


## 11. Calcular os resumos regionais e nacional

Para resumir uma região, agrupamos diretamente as notas sorteadas de suas UFs. Cada nota tem o mesmo peso; UFs com mais participantes contribuem com mais notas. Não calculamos a média ou a mediana das medianas estaduais. O resumo nacional também parte das notas, por ano e área.

Como o sorteio é de aproximadamente 10% em cada grupo, a composição da amostra acompanha aproximadamente a dos elegíveis com UF conhecida. O arredondamento gera pequenas diferenças nessa fração. Uma inferência populacional exigiria considerar pesos e incerteza da amostragem. Aqui, o resumo nacional continua restrito aos participantes com UF da escola informada.


In [ ]:
#calculo as mesmas medidas juntando as UFs de cada região
colunas_regiao = ["ANO", "REGIAO", "AREA"]
grupos_regiao = dados_analise.groupby(colunas_regiao, observed=True)["NOTA"]

resumo_regiao = grupos_regiao.agg(
    MEDIA="mean",
    MEDIANA="median",
    N="count",
    DESVIO_PADRAO="std",
    ASSIMETRIA="skew",
).reset_index()

percentis_regiao = (
    grupos_regiao.quantile([0.05, 0.25, 0.75, 0.95])
    .unstack()
    .rename(columns={0.05: "P5", 0.25: "Q1", 0.75: "Q3", 0.95: "P95"})
    .reset_index()
)

resumo_regiao = resumo_regiao.merge(percentis_regiao, on=colunas_regiao)
resumo_regiao["AMPLITUDE_INTERQUARTIL"] = resumo_regiao["Q3"] - resumo_regiao["Q1"]
resumo_regiao["NOTA_REPRESENTATIVA"] = resumo_regiao["MEDIANA"]

#resumo nacional por ano e área
resumo_brasil = (
    dados_analise.groupby(["ANO", "AREA"], observed=True)["NOTA"]
    .agg(MEDIA="mean", MEDIANA="median", N="count", DESVIO_PADRAO="std")
    .reset_index()
)


## 12. Distribuição das notas por estado, área e ano

Escolha UF, ano e área para ver onde as notas se concentram. O histograma mostra percentuais; as linhas marcam a média e a mediana. O filtro de faixas muda a aparência das barras, sem alterar as medidas.

Marque **Ver tabela detalhada** para consultar tamanho da amostra, elegíveis, medidas centrais, dispersão e extremos. Todas as notas do grupo entram. P5 e P95 descrevem a dispersão das notas, não um intervalo de confiança.


In [ ]:
def mostrar_distribuicao(ano, area, uf_escola, faixas, ver_tabela=False):
    selecao = (
        (dados_analise["ANO"] == ano)
        & (dados_analise["AREA"] == area)
        & (dados_analise["UF_ESCOLA"] == uf_escola)
    )
    notas = dados_analise.loc[selecao, "NOTA"]
    if notas.empty:
        print("Não há notas neste recorte.")
        return
    limites = np.linspace(notas.min(), notas.max() + 0.001, faixas + 1)
    frequencias, limites = np.histogram(notas, bins=limites)
    tabela = pd.DataFrame(
        {
            "Centro da faixa": (limites[:-1] + limites[1:]) / 2,
            "Percentual": frequencias / len(notas) * 100,
            "Início": limites[:-1],
            "Fim": limites[1:],
        }
    )
    grafico = px.bar(
        tabela,
        x="Centro da faixa",
        y="Percentual",
        hover_data=["Início", "Fim"],
        title=f"Distribuição das notas<br><sup>{nome_por_uf[uf_escola]} · {area} · {ano}</sup>",
    )
    grafico.update_traces(width=(limites[1] - limites[0]) * 0.95)
    grafico.add_vline(x=notas.mean(), line_color="#D55E00", line_dash="dot")
    grafico.add_vline(x=notas.median(), line_color="#009E73", line_dash="dash")
    grafico.update_layout(height=480, margin=dict(t=95, b=65, l=70, r=35))
    grafico.update_xaxes(title="Nota (pontos)")
    grafico.update_yaxes(title="Notas na faixa (%)")
    exibir_grafico(grafico)
    display(
        Markdown(
            f"**{len(notas):,} notas** · Média: **{notas.mean():.2f}** "
            f"· Mediana: **{notas.median():.2f}** pontos.  \n"
            "Laranja pontilhado: média. Verde tracejado: mediana."
        )
    )
    selecao_resumo = (
        (resumo_uf["ANO"] == ano)
        & (resumo_uf["AREA"] == area)
        & (resumo_uf["UF_ESCOLA"] == uf_escola)
    )
    tabela_resumo = resumo_uf[selecao_resumo]
    if ver_tabela:
        exibir_tabela(
            tabela_resumo[
                [
                    "N",
                    "REGISTROS_ELEGIVEIS",
                    "MEDIA",
                    "MEDIANA",
                    "DESVIO_PADRAO",
                    "ASSIMETRIA",
                    "P5",
                    "Q1",
                    "Q3",
                    "P95",
                    "MINIMO",
                    "MAXIMO",
                ]
            ].round(2)
        )


montar_painel(
    mostrar_distribuicao,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    uf_escola=widgets.Dropdown(
        options=sorted(ufs_validas), value="SP", description="UF escola:"
    ),
    faixas=widgets.IntSlider(
        min=10,
        max=80,
        step=10,
        value=40,
        description="Faixas:",
        continuous_update=False,
    ),
)


## 15. Diagnóstico opcional: Shapiro–Wilk e gráfico Q–Q

Escolha uma prova objetiva e marque **Executar**. Usamos até 5.000 notas, sorteadas com a mesma semente, no teste e no gráfico Q–Q. Esse limite segue a ressalva do SciPy sobre a precisão do p-valor acima de 5.000 observações; não garante todos os pressupostos do teste.

W é a estatística do Shapiro–Wilk. Um p-valor menor que 0,05 indica evidência contra normalidade, sob as hipóteses do teste. Um valor maior ou igual a 0,05 indica que o teste não a rejeitou; não prova que a distribuição seja normal. O p-valor também não é a probabilidade de normalidade nem a confiança da mediana.

No Q–Q, pontos próximos da linha sugerem aproximação ao padrão normal. Curvas e desvios nas pontas ajudam a observar diferenças e caudas.

O diagnóstico é exploratório: amostras grandes detectam desvios pequenos, há notas arredondadas e empatadas, e participantes compartilham escolas. Assim, independência e continuidade são aproximações. Redação fica fora deste teste por sua escala discreta e seus muitos empates, mas permanece nos outros painéis. Repetir o teste em várias UFs não equivale a compará-las estatisticamente.

**A mediana continua sendo a medida principal, qualquer que seja o resultado.**


In [ ]:
def mostrar_normalidade(ano, area, uf_escola, executar):
    if not executar:
        print("Marque Executar para ver o diagnóstico e o gráfico Q–Q.")
        return
    selecao = (
        (dados_analise["ANO"] == ano)
        & (dados_analise["AREA"] == area)
        & (dados_analise["UF_ESCOLA"] == uf_escola)
    )
    notas = dados_analise.loc[selecao, "NOTA"]
    if len(notas) < 3 or notas.nunique() < 2:
        print("Quantidade ou variação insuficiente para o diagnóstico.")
        return
    notas_teste = notas.sample(n=min(5000, len(notas)), random_state=SEMENTE)
    resultado = shapiro(notas_teste)
    exibir_tabela(
        pd.DataFrame(
            {
                "N total do recorte": [len(notas)],
                "N no diagnóstico": [len(notas_teste)],
                "W de Shapiro": [resultado.statistic],
                "p-valor": [resultado.pvalue],
            }
        )
    )
    if resultado.pvalue < 0.05:
        print("Evidência contra normalidade (5%).")
    else:
        print("O teste não rejeitou normalidade (5%); isso não a comprova.")
    print("Diagnóstico exploratório; a medida principal continua sendo a mediana.")
    #compara quantis observados com quantis teóricos de uma distribuição normal
    (quantis_normais, notas_ordenadas), (inclinacao, intercepto, correlacao) = probplot(
        notas_teste, dist="norm"
    )
    grafico = go.Figure()
    grafico.add_scatter(
        x=quantis_normais,
        y=notas_ordenadas,
        mode="markers",
        name="Notas observadas",
        marker=dict(size=4),
    )
    grafico.add_scatter(
        x=quantis_normais,
        y=inclinacao * quantis_normais + intercepto,
        mode="lines",
        name="Referência normal",
    )
    grafico.update_layout(
        title=f"Comparação com a distribuição normal<br><sup>{uf_escola} · {area} · {ano}</sup>",
        xaxis_title="Quantil teórico normal",
        yaxis_title="Nota observada",
        height=470,
        margin=dict(t=100, b=110, l=80, r=35),
        legend=dict(orientation="h", y=-0.22),
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_normalidade,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS[:-1],
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    uf_escola=widgets.Dropdown(
        options=sorted(ufs_validas), value="SP", description="UF escola:"
    ),
    executar=widgets.Checkbox(value=False, description="Executar"),
)


## 16. Mapa da mediana por UF da escola

O código do IBGE liga a tabela ao desenho de cada estado. A cor representa a mediana. Ao mudar o ano, mantemos a escala de cor da área para permitir a comparação no tempo.

Passe o cursor para consultar tamanho da amostra, média, mediana e quartis. A UF é a da escola, não a da residência ou da prova. Uma mediana maior descreve esse recorte; não mede a qualidade de todo o sistema educacional.

Use `malha_ufs_plotly.geojson`, que já tem a orientação dos polígonos preparada para o Plotly. Sem esse arquivo, os outros painéis continuam funcionando.


In [ ]:
#sem malha, apenas o painel do mapa fica indisponível.
mapa_brasil = None
if ARQUIVO_MAPA.exists():
    with open(ARQUIVO_MAPA, encoding="utf-8") as arquivo:
        mapa_brasil = json.load(arquivo)

#manntém a mesma escala de cor ao trocar o ano dentro de cada área
limites_cores = resumo_uf.groupby("AREA", observed=True)["MEDIANA"].agg(["min", "max"])


def mostrar_mapa(ano, area):
    if mapa_brasil is None:
        print(
            "Coloque malha_ufs_plotly.geojson na pasta e execute esta célula novamente. Os outros painéis funcionam sem a malha."
        )
        return
    dados_mapa = resumo_uf[
        (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    ].copy()

    grafico = px.choropleth(
        dados_mapa,
        geojson=mapa_brasil,
        locations="COD_UF_IBGE",
        featureidkey="properties.codarea",
        color="MEDIANA",
        color_continuous_scale="Cividis",
        range_color=(limites_cores.loc[area, "min"], limites_cores.loc[area, "max"]),
        hover_name="NOME_UF",
        hover_data={
            "COD_UF_IBGE": False,
            "UF_ESCOLA": True,
            "REGIAO": True,
            "MEDIANA": ":.2f",
            "MEDIA": ":.2f",
            "Q1": ":.2f",
            "Q3": ":.2f",
            "N": ":,.0f",
        },
        labels={
            "UF_ESCOLA": "UF da escola",
            "REGIAO": "Região",
            "MEDIANA": "Mediana",
            "MEDIA": "Média",
            "Q1": "1º quartil",
            "Q3": "3º quartil",
            "N": "Notas na amostra",
        },
    )

    grafico.update_geos(fitbounds="geojson", projection_type="mercator", visible=False)
    grafico.update_traces(marker_line_color="white", marker_line_width=0.7)
    grafico.update_layout(
        title={
            "text": (
                "<b>Desempenho típico por UF da escola</b><br>"
                f"<span style='font-size:17px'>{area} — {ano}</span><br>"
                "<span style='font-size:13px;color:#555'>Cor baseada na mediana</span>"
            ),
            "x": 0.5,
            "xanchor": "center",
        },
        coloraxis_colorbar_title="Mediana",
        height=570,
        margin={"l": 10, "r": 20, "t": 110, "b": 15},
        paper_bgcolor="white",
        font={"family": "Arial, sans-serif", "color": "#222"},
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_mapa,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="330px"),
    ),
)


## 17. Evolução regional: uma área ou todas

Cada ponto mostra a mediana das notas da região em um ano e área. Escolha o período e compare as regiões ou selecione apenas uma. A visualização começa em Matemática. A opção **Todas** abre um painel por área, com eixos próprios.

As linhas ligam amostras de pessoas diferentes. Mudanças podem refletir composição dos participantes, cobertura escolar e contexto do exame. As escalas de Redação e das provas objetivas não devem ser interpretadas como equivalentes.


In [ ]:
def mostrar_evolucao(area, regiao, periodo):
    tabela = resumo_regiao[resumo_regiao["ANO"].between(periodo[0], periodo[1])].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    if area != "Todas":
        tabela = tabela[tabela["AREA"] == area]
    tabela = tabela.sort_values("ANO")
    grafico = px.line(
        tabela,
        x="ANO",
        y="MEDIANA",
        color="REGIAO",
        markers=True,
        facet_col="AREA" if area == "Todas" else None,
        facet_col_spacing=0.12,
        facet_col_wrap=2 if area == "Todas" else 0,
        color_discrete_map=cores_regioes,
        category_orders={"AREA": AREAS, "REGIAO": REGIOES},
        hover_data={"N": True, "Q1": ":.1f", "Q3": ":.1f"},
        labels={"MEDIANA": "Mediana (pontos)", "ANO": "Ano", "REGIAO": "Região"},
        title=f"Evolução das medianas regionais<br><sup>{area}</sup>",
    )
    grafico.update_xaxes(dtick=1, showticklabels=True)
    grafico.update_yaxes(matches=None, showticklabels=True)
    grafico.update_layout(
        height=940 if area == "Todas" else 500,
        margin=dict(t=110, b=120, l=75, r=40),
        legend=dict(orientation="h", y=-0.12),
    )
    # Deixa só o nome da área no título de cada painel.
    for anotacao in grafico.layout.annotations:
        anotacao.text = anotacao.text.replace("AREA=", "")
    exibir_grafico(grafico)


montar_painel(
    mostrar_evolucao,
    area=widgets.Dropdown(
        options=["Todas"] + AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    periodo=widgets.IntRangeSlider(
        min=2020,
        max=2024,
        value=[2020, 2024],
        step=1,
        description="Período:",
        continuous_update=False,
    ),
)


## 18. Ranking de UFs no Brasil ou em uma região

Escolha ano, área e região para ordenar as UFs pela mediana. Empates recebem a mesma posição. A ordem usa a precisão do resultado, embora o gráfico mostre duas casas decimais. A tabela mantém tamanho da amostra, elegíveis, média e quartis.

O ranking descreve a amostra. A primeira posição não prova superioridade estatística ou melhor educação. Diferenças pequenas podem mudar em outro sorteio, e ainda não calculamos intervalos de confiança. Compare também a média e a distribuição das notas.


In [ ]:
def mostrar_ranking(ano, area, regiao, ordem, quantidade, ver_tabela=False):
    selecao = (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    tabela = resumo_uf[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    tabela["Posição no recorte"] = (
        tabela["MEDIANA"].rank(method="min", ascending=False).astype(int)
    )
    menores_primeiro = ordem == "Menores"
    tabela = tabela.sort_values(
        ["MEDIANA", "UF_ESCOLA"], ascending=[menores_primeiro, True]
    ).head(quantidade)
    grafico = px.bar(
        tabela.sort_values("MEDIANA"),
        x="MEDIANA",
        y="UF_ESCOLA",
        orientation="h",
        color="REGIAO",
        color_discrete_map=cores_regioes,
        text="MEDIANA",
        hover_data={"N": True, "Q1": ":.2f", "Q3": ":.2f", "Posição no recorte": True},
        labels={
            "UF_ESCOLA": "UF da escola",
            "MEDIANA": "Mediana (pontos)",
            "REGIAO": "Região",
        },
        title=f"{ordem} medianas por UF da escola<br><sup>{area} · {ano} · {regiao}</sup>",
    )
    grafico.update_traces(
        texttemplate="%{text:.2f}", textposition="outside", cliponaxis=False
    )
    grafico.update_layout(
        height=max(470, 27 * len(tabela) + 200),
        margin=dict(t=100, r=100, b=110, l=75),
        legend=dict(orientation="h", y=-0.12, title_text=""),
    )
    grafico.update_xaxes(range=[0, tabela["MEDIANA"].max() * 1.18])
    grafico.update_yaxes(categoryorder="total ascending", title=None)
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(
            tabela[
                [
                    "Posição no recorte",
                    "UF_ESCOLA",
                    "NOME_UF",
                    "REGIAO",
                    "N",
                    "REGISTROS_ELEGIVEIS",
                    "MEDIANA",
                    "MEDIA",
                    "Q1",
                    "Q3",
                ]
            ].round(2)
        )


montar_painel(
    mostrar_ranking,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    ordem=widgets.Dropdown(options=["Maiores", "Menores"], description="Ordem:"),
    quantidade=widgets.IntSlider(
        min=1, max=27, value=10, description="Exibir:", continuous_update=False
    ),
)


## 19. Evolução dos estados em cada região

Escolha região, área e período para acompanhar as medianas estaduais. Marque **Ver tabela detalhada** para consultar todas as UFs empatadas na maior mediana de cada ano.

São amostras de participantes diferentes ao longo do tempo. Mudanças nas linhas podem refletir composição, cobertura e distribuição das notas; o gráfico sozinho não permite atribuir causas.


In [ ]:
def mostrar_estados(regiao, area, periodo, ver_tabela=False):
    selecao = (
        (resumo_uf["REGIAO"] == regiao)
        & (resumo_uf["AREA"] == area)
        & resumo_uf["ANO"].between(periodo[0], periodo[1])
    )
    tabela = resumo_uf[selecao].sort_values("ANO").copy()
    grafico = px.line(
        tabela,
        x="ANO",
        y="MEDIANA",
        color="UF_ESCOLA",
        markers=True,
        hover_data={"N": True, "REGISTROS_ELEGIVEIS": True},
        labels={"ANO": "Ano", "MEDIANA": "Mediana (pontos)", "UF_ESCOLA": "UF escolar"},
        title=f"Medianas dos estados · {regiao}<br><sup>{area}</sup>",
    )
    grafico.update_xaxes(dtick=1)
    grafico.update_layout(
        height=500,
        margin=dict(t=100, b=100, l=75, r=35),
        legend=dict(orientation="h", y=-0.2, title_text=""),
    )
    exibir_grafico(grafico)
    maior_por_ano = tabela.groupby("ANO")["MEDIANA"].transform("max")
    #mostra  todas as UFs empatadas na maior mediana de cada ano
    destaques = tabela.loc[
        tabela["MEDIANA"] == maior_por_ano,
        ["ANO", "UF_ESCOLA", "MEDIANA", "N"],
    ]
    if ver_tabela:
        exibir_tabela(destaques.round(2))


montar_painel(
    mostrar_estados,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    regiao=widgets.Dropdown(options=REGIOES, value="Sudeste", description="Região:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    periodo=widgets.IntRangeSlider(
        min=2020,
        max=2024,
        value=[2020, 2024],
        description="Período:",
        continuous_update=False,
    ),
)


## 20. Posição das regiões em cada área

Cada coluna mostra uma área. A cor indica a posição da região pela mediana dentro daquela coluna: 1 é a maior. Empates recebem a mesma posição. O número escrito é a mediana em pontos, e o cursor mostra a posição.

As posições permitem comparar a ordem das regiões sem misturar as escalas das áreas. A cor não mostra o tamanho da diferença: confira as medianas, pois posições consecutivas podem estar próximas ou distantes em pontos.


In [ ]:
def mostrar_painel_areas(ano):
    tabela = resumo_regiao[resumo_regiao["ANO"] == ano]
    tabela = tabela.pivot(index="REGIAO", columns="AREA", values="MEDIANA")
    tabela = tabela.reindex(index=REGIOES, columns=AREAS)
    posicoes = tabela.rank(axis=0, ascending=False, method="min")
    grafico = go.Figure(
        go.Heatmap(
            z=posicoes.to_numpy(),
            x=AREAS,
            y=REGIOES,
            text=tabela.round(2).to_numpy(),
            texttemplate="%{text:.2f}",
            colorscale="Blues_r",
            zmin=1,
            zmax=5,
            colorbar=dict(title="Posição", tickvals=[1, 2, 3, 4, 5], thickness=14),
            xgap=4,
            ygap=4,
            textfont=dict(size=16),
            hovertemplate="%{y}<br>%{x}<br>Mediana: %{text:.2f}<br>Posição: %{z}<extra></extra>",
        )
    )
    grafico.update_layout(
        title=f"Regiões em cada área · {ano}<br><sup>Números: medianas · cores: posição na área</sup>",
        height=480,
        margin=dict(t=100, b=100, l=120, r=70),
    )
    grafico.update_xaxes(
        tickvals=AREAS,
        ticktext=[
            "Linguagens",
            "Ciências<br>Humanas",
            "Ciências da<br>Natureza",
            "Matemática",
            "Redação",
        ],
        tickangle=0,
    )
    grafico.update_yaxes(autorange="reversed")
    exibir_grafico(grafico)


montar_painel(
    mostrar_painel_areas,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
)


## 21. Quantos alcançam uma nota escolhida?

Escolha ano, área, região e uma nota de referência. Para cada UF, calculamos a porcentagem da amostra com nota maior ou igual ao limite. O denominador é o tamanho da amostra daquela UF, mostrado na tabela e no cursor.

O limite serve para explorar os dados; não é uma nota oficial de aprovação ou proficiência. Como as áreas têm escalas diferentes, usar o mesmo limite entre elas exige justificativa.


In [ ]:
def mostrar_percentual(ano, area, regiao, limite, ver_tabela=False):
    selecao = (dados_analise["ANO"] == ano) & (dados_analise["AREA"] == area)
    tabela = dados_analise[selecao].copy()
    if regiao != "Todas":
        tabela = tabela[tabela["REGIAO"] == regiao]
    tabela["ATINGIU"] = tabela["NOTA"] >= limite
    grupos = tabela.groupby("UF_ESCOLA", observed=True)["ATINGIU"]
    resumo = grupos.agg(N="count", Quantidade="sum").reset_index()
    resumo["Percentual"] = 100 * resumo["Quantidade"] / resumo["N"]
    grafico = px.bar(
        resumo.sort_values("Percentual"),
        x="Percentual",
        y="UF_ESCOLA",
        orientation="h",
        text="Percentual",
        hover_data=["N", "Quantidade"],
        title=f"Notas a partir de {limite} pontos<br><sup>{area} · {ano} · {regiao}</sup>",
        labels={"Percentual": "Parcela da amostra (%)", "UF_ESCOLA": "UF escolar"},
    )
    grafico.update_traces(
        texttemplate="%{text:.1f}%", textposition="outside", cliponaxis=False
    )
    grafico.update_xaxes(range=[0, 110])
    grafico.update_yaxes(title=None)
    grafico.update_layout(
        height=max(420, 27 * len(resumo) + 120), margin=dict(t=100, r=75, b=65, l=75)
    )
    exibir_grafico(grafico)
    if ver_tabela:
        exibir_tabela(resumo.sort_values("Percentual", ascending=False).round(2))


montar_painel(
    mostrar_percentual,
    ver_tabela=widgets.Checkbox(value=False, description="Ver tabela detalhada"),
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    regiao=widgets.Dropdown(options=["Todas"] + REGIOES, description="Região:"),
    limite=widgets.IntSlider(
        min=0,
        max=1000,
        step=20,
        value=600,
        description="Nota ≥",
        continuous_update=False,
    ),
)


## 22. Mais participantes estão associados a maior mediana?

Cada ponto representa uma UF. O eixo horizontal mostra os elegíveis com UF da escola, contados antes do sorteio; o vertical mostra a mediana da amostra. A cor identifica a região, e o cursor mostra o estado. A escala logarítmica ajuda a visualizar UFs com volumes menores.

O gráfico permite observar uma possível associação entre volume e mediana. Ele não demonstra causa nem explica a participação por população, renda ou cultura sem outros dados.


In [ ]:
def mostrar_volume_nota(ano, area, escala_log):
    selecao = (resumo_uf["ANO"] == ano) & (resumo_uf["AREA"] == area)
    tabela = resumo_uf[selecao]
    grafico = px.scatter(
        tabela,
        x="REGISTROS_ELEGIVEIS",
        y="MEDIANA",
        color="REGIAO",
        hover_name="UF_ESCOLA",
        color_discrete_map=cores_regioes,
        log_x=escala_log,
        hover_data={"N": True, "NOME_UF": True},
        labels={
            "REGISTROS_ELEGIVEIS": "Elegíveis com UF escolar",
            "MEDIANA": "Mediana (pontos)",
            "REGIAO": "Região",
        },
        title=f"Participação e mediana das notas<br><sup>{area} · {ano}</sup>",
    )
    grafico.update_traces(marker=dict(size=10))
    grafico.update_layout(
        height=520,
        margin=dict(t=100, b=110, l=80, r=40),
        legend=dict(orientation="h", y=-0.22, title_text=""),
    )
    exibir_grafico(grafico)


montar_painel(
    mostrar_volume_nota,
    ano=widgets.Dropdown(options=ANOS, value=2024, description="Ano:"),
    area=widgets.Dropdown(
        options=AREAS,
        value="Matemática",
        description="Área:",
        layout=widgets.Layout(width="350px"),
    ),
    escala_log=widgets.Checkbox(value=True, description="Volume em escala logarítmica"),
)


## 23. Preparar tabelas para as próximas etapas

Deixamos prontos os resumos por UF, região e Brasil, com uma linha por ano e área em cada território. A base estadual também informa a fração amostral e a medida principal. Ative `EXPORTAR_TABELAS` para salvar os três CSVs em `tabelas_descritivas`.

Mantemos as medidas, as contagens e as chaves territoriais. O Shapiro não entra como atributo para prever nota, pois é calculado a partir da própria distribuição das notas.

Um modelo com unidade UF × ano terá apenas 135 combinações nos cinco anos. As áreas não são cinco observações territoriais independentes. A cobertura disponível é nacional por ano e área; não podemos apresentá-la como cobertura de cada UF, pois desconhecemos a UF de quem não a informou.


In [ ]:
base_proxima_etapa = resumo_uf.copy()
base_proxima_etapa["FRACAO_AMOSTRAL"] = (
    base_proxima_etapa["N"] / base_proxima_etapa["REGISTROS_ELEGIVEIS"]
)
base_proxima_etapa["MEDIDA_PRINCIPAL"] = "Mediana da amostra"
assert not base_proxima_etapa.duplicated(["ANO", "UF_ESCOLA", "AREA"]).any()
assert not resumo_regiao.duplicated(["ANO", "REGIAO", "AREA"]).any()
print("Base estadual pronta:", base_proxima_etapa.shape)
print("Base regional pronta:", resumo_regiao.shape)

if EXPORTAR_TABELAS:
    pasta_exportacao = PASTA_DADOS / "tabelas_descritivas"
    pasta_exportacao.mkdir(exist_ok=True)
    base_proxima_etapa.to_csv(
        pasta_exportacao / "enem_descritiva_uf_escola.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    resumo_regiao.to_csv(
        pasta_exportacao / "enem_descritiva_regiao.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    resumo_brasil.to_csv(
        pasta_exportacao / "enem_descritiva_amostra_nacional.csv",
        sep=";",
        encoding="utf-8-sig",
        index=False,
    )
    print("Tabelas salvas em:", pasta_exportacao)
else:
    print(
        "Tabelas disponíveis na memória. Ative EXPORTAR_TABELAS na configuração para salvar CSVs."
    )


Base estadual pronta: (675, 22)
Base regional pronta: (125, 14)
Tabelas disponíveis na memória. Ative EXPORTAR_TABELAS na configuração para salvar CSVs.




# Etapa 2: Desigualdades educacionais e condições dos territórios

Na primeira etapa, organizamos as notas e descrevemos suas diferenças. Agora queremos saber **quais condições econômicas e escolares acompanham essas diferenças** e quais territórios merecem investigação mais aprofundada pelo poder público.

A finalidade do projeto é apoiar o diagnóstico de necessidades educacionais. Uma secretaria pode usar os resultados para selecionar territórios para visitas técnicas, verificar carências e estudar experiências. O ENEM retrata seus participantes, não todos os estudantes; por isso, este estudo não decide sozinho a distribuição de recursos.

Seguiremos a mesma pergunta do começo ao fim: **onde estão as diferenças, que características aparecem junto delas e quais informações podem ajudar a antecipá-las na etapa 3?** A comparação abrange 27 UFs, cinco regiões, cinco áreas e as edições de 2020 a 2024. As notas de áreas diferentes não serão somadas em uma nota geral.

| Passo | O que queremos entender | O que fica para a próxima etapa |
|---|---|---|
| 1. Conferir o recorte | Quem aparece nas notas e o que mudou entre anos? | Contagens e limites de cobertura |
| 2. Examinar a distribuição | Média, mediana e possíveis picos contam a mesma história? | Definição do resultado a estimar |
| 3. Integrar as fontes | Como juntar notas, escolas e condições econômicas sem duplicar linhas? | Base territorial auditada |
| 4. Comparar contextos | Renda, infraestrutura e oferta de ensino acompanham o desempenho? | Atributos candidatos e hipóteses |
| 5. Investigar mudanças | A composição dos participantes ajuda a entender a evolução? | Limites para interpretar quedas |
| 6. Preparar a continuidade | Quais informações já existiam antes do resultado? | Dataset com atributos anteriores e testes de viabilidade; modelo final na etapa 3 |


**Nesta revisão:** primeiro conferimos os dados e as explicações possíveis; depois comparamos associações lineares e não lineares; por fim fazemos ajustes pequenos para verificar contribuição conjunta, lags e erros em anos posteriores. Os testes podem rejeitar uma ideia inicial. Não escolheremos um estado nem um conjunto de atributos só porque parecem interessantes.

Cada célula alterada ou acrescentada está marcada com **AQUI TEVE MELHORIA**. Os gráficos já salvos mostram os filtros iniciais; para trocar filtros, execute as células desta etapa em ordem. Os novos resultados estaduais usam todas as notas elegíveis, mantendo a amostra para examinar distribuições.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
from scipy.stats import gaussian_kde, spearmanr
from scipy.signal import find_peaks


PASTA_ESTUDO = Path('/content/drive/MyDrive/ENEM/estudo_enem_revisado')
if not (PASTA_ESTUDO / 'dados').exists():
    for pasta in [Path.cwd(), Path.cwd() / 'output/estudo_enem_revisado', Path('/content/estudo_enem_revisado')]:
        if (pasta / 'dados/amostra_notas_contexto.parquet').exists():
            PASTA_ESTUDO = pasta
            break
if not (PASTA_ESTUDO / 'dados').exists():
    raise FileNotFoundError('Extraia a pasta estudo_enem_revisado e ajuste PASTA_ESTUDO. No Colab, monte o Drive antes.')

PASTA_ENTRADA = PASTA_ESTUDO / 'dados'
PASTA_RESULTADOS = PASTA_ESTUDO / 'resultados'
PASTA_RESULTADOS.mkdir(exist_ok=True)
MOSTRAR_FILTROS = True

def ler(nome):
    return pd.read_csv(PASTA_ENTRADA / nome, sep=';')

def salvar(tabela, nome):
    tabela.to_csv(PASTA_RESULTADOS / nome, sep=';', index=False, encoding='utf-8-sig')

enem_uf = ler('enem_uf_2020_2024.csv')
enem_regiao = ler('enem_regiao_2020_2024.csv')
enem_nacional = ler('enem_nacional_2020_2024.csv')
enem_redes = ler('enem_rede_local_2020_2024.csv')
enem_estadual_completo = ler('enem_estadual_completo_2020_2024.csv')
cobertura = ler('cobertura_uf_escola.csv')
amostra = pd.read_parquet(PASTA_ENTRADA / 'amostra_notas_contexto.parquet')
escolas = pd.read_parquet(PASTA_ENTRADA / 'escolas_ensino_medio.parquet')
perfis = pd.read_parquet(PASTA_ENTRADA / 'perfis_participantes.parquet')

ANOS_ESTUDO = list(range(2020, 2025))
AREAS_ESTUDO = ['Linguagens', 'Ciências Humanas', 'Ciências da Natureza', 'Matemática', 'Redação']
REGIOES_ESTUDO = ['Norte', 'Nordeste', 'Centro-Oeste', 'Sudeste', 'Sul']
UFS_ESTUDO = sorted(enem_uf.UF_ESCOLA.unique())
TERRITORIOS = ['Brasil (UF informada)'] + REGIOES_ESTUDO + UFS_ESTUDO
REDES_ESTUDO = ['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada', 'Não informada']
CORES_REGIOES = dict(zip(REGIOES_ESTUDO, ['#2276A5', '#BF7B13', '#298574', '#C95C42', '#86599D']))

print('Pasta:', PASTA_ESTUDO)
print('Período: 2020–2024 | 27 UFs | 5 áreas | testes de viabilidade; nenhum modelo final')

Pasta: /content/drive/MyDrive/ENEM/estudo_enem_revisado
Período: 2020–2024 | 27 UFs | 5 áreas | testes de viabilidade; nenhum modelo final




## 24. Como as fontes entram na investigação

A unidade principal será **UF × ano × área**. Por exemplo: Ceará, 2022, Matemática. O Censo começa com uma linha por escola, mas primeiro resumimos suas informações para cada UF e ano. Só depois ligamos ao ENEM. A renda do IBGE também descreve a UF, não a família de cada participante.

| Fonte | Informação utilizada | Por que entra | Chave da junção |
|---|---|---|---|
| [ENEM — Inep](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) | Notas, presença, UF e rede | Resultado educacional e composição dos participantes | UF + ano + área |
| [Censo Escolar — Inep](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar) | Recursos, matrículas e tempo integral | Condições de oferta do ensino médio | UF + ano; rede no recorte estadual |
| [PNAD Contínua — IBGE, tabela 7531](https://sidra.ibge.gov.br/tabela/7531) | Renda domiciliar por pessoa | Contexto socioeconômico | UF + ano |
| [Indicadores educacionais — Inep](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais) | Formação docente, horas-aula e rendimento | Condições de ensino e permanência | UF + ano; rede quando disponível |

As três primeiras bases cobrem todo o período e formam o núcleo da integração. A variável 10824 da PNAD informa rendimento médio mensal real domiciliar por pessoa, a preços médios do próprio ano. O [IPCA do IBGE](https://sidra.ibge.gov.br/tabela/1737) é usado aqui para reexpressar esses valores em preços de 2024: é um ajuste pelo índice nacional, não uma medida do custo de vida de cada UF. Conservamos a versão recebida e a versão reexpressa para conferência.

**O vínculo aqui é territorial.** `SG_UF_ESC` no ENEM e `SG_UF` no Censo foram padronizadas como `UF_ESCOLA`. Isso não significa que localizamos a escola frequentada por cada participante. As escolas do Censo representam a oferta de ensino médio da UF. Não usamos nome, nota ou posição de linha para tentar acompanhar pessoas entre anos.

Abaixo conferimos a disponibilidade das fontes antes de qualquer associação. Uma chave presente não garante que todos os campos estejam preenchidos; essa segunda verificação aparece na exportação.

O FUNDEB disponível apenas para dezembro foi retirado da junção principal: um único mês não representa gasto anual nem exposição de cada estudante. A justificativa de exclusão aparece na seção 40. As três bases centrais continuam sendo ENEM, Censo Escolar e PNAD, complementadas por indicadores educacionais.


In [ ]:
censo_uf = ler('censo_uf_2020_2024.csv')
renda_uf = ler('renda_uf_2020_2024.csv')
formacao = ler('indicador_afd_uf_rede.csv')
jornada = ler('indicador_had_uf_rede.csv')
rendimento = ler('indicador_rendimento_uf_rede.csv')

fontes_anuais = {'ENEM': enem_uf, 'Censo Escolar': censo_uf, 'IBGE/PNAD': renda_uf,
                'Formação docente': formacao, 'Horas-aula': jornada, 'Rendimento': rendimento}
disponibilidade = pd.DataFrame({nome: tabela.groupby('ANO').UF_ESCOLA.nunique()
                               for nome, tabela in fontes_anuais.items()}).reindex(ANOS_ESTUDO)
assert disponibilidade.eq(27).all().all(), 'Confira as UFs ausentes antes de seguir.'
display(disponibilidade.rename_axis('Ano').style.set_caption('Número de UFs com dados em cada fonte'))
salvar(disponibilidade.reset_index(), 'cobertura_fontes.csv')

,ENEM,Censo Escolar,IBGE/PNAD,Formação docente,Horas-aula,Rendimento
Ano,,,,,,
2020,27,27,27,27,27,27
2021,27,27,27,27,27,27
2022,27,27,27,27,27,27
2023,27,27,27,27,27,27
2024,27,27,27,27,27,27


## 25. Preparar gráficos e filtros

Os próximos painéis usam os mesmos controles de ano, área e território. Os gráficos terão tamanho fixo e legível, sem caixas de saída com uma altura mínima exagerada. Cada filtro atualiza apenas seu painel. As tabelas detalhadas ficam opcionais.

Nas comparações nacionais e regionais, agregamos as notas ou os registros originais do recorte: **não calculamos a média das medianas dos estados**. “Públicas” reúne federal, estadual e municipal; rede não informada fica identificada, sem ser atribuída à rede pública.

In [ ]:
plt.rcParams.update({'figure.figsize': (10, 4.8), 'figure.dpi': 110,
    'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'text.color': '#243747', 'axes.labelcolor': '#243747', 'savefig.dpi': 150})

def escolher(opcoes, valor, descricao):
    return widgets.Dropdown(options=opcoes, value=valor, description=descricao,
        style={'description_width': '100px'}, layout=widgets.Layout(width='360px'))

def periodo():
    return widgets.IntRangeSlider(value=[2020, 2024], min=2020, max=2024,
        description='Período:', continuous_update=False, layout=widgets.Layout(width='360px'))

def painel(funcao, **controles):
    if not MOSTRAR_FILTROS:
        funcao(**{nome: controle.value for nome, controle in controles.items()})
        return
    saida = widgets.interactive_output(funcao, controles)
    linha = widgets.HBox(list(controles.values()), layout=widgets.Layout(flex_flow='row wrap'))
    display(widgets.VBox([linha, saida]))

def recortar(tabela, territorio='Brasil (UF informada)', rede='Todas'):
    if territorio in UFS_ESTUDO:
        tabela = tabela[tabela.UF_ESCOLA.eq(territorio)]
    elif territorio in REGIOES_ESTUDO:
        tabela = tabela[tabela.REGIAO.eq(territorio)]
    if rede == 'Públicas':
        tabela = tabela[tabela.REDE.isin(['Federal', 'Estadual', 'Municipal'])]
    elif rede != 'Todas':
        tabela = tabela[tabela.REDE.eq(rede)]
    return tabela

def terminar(fig, nome):
    fig.tight_layout()
    # A imagem exportada acompanha os filtros escolhidos por último.
    fig.savefig(PASTA_RESULTADOS / (nome + '.png'), bbox_inches='tight')
    plt.show()
    plt.close(fig)

def tabela_opcional(dados, mostrar):
    if mostrar:
        display(dados.round(2).reset_index(drop=True).style.hide(axis='index'))

def grade(ax):
    ax.grid(axis='y', alpha=.18)
    ax.set_axisbelow(True)

## 26. Antes de comparar resultados: quem conseguimos observar?

**Pergunta:** a parcela de participantes com UF escolar conhecida permaneceu a mesma? A linha à esquerda mostra a cobertura. À direita, comparamos a média de quem tem UF conhecida com a média dos demais elegíveis. A segunda é recuperada pela diferença das somas e das contagens; é uma aproximação por arredondamento dos resumos nacionais.

Mantemos a elegibilidade da etapa 1: presença na prova e nota disponível; Redação exige situação regular. A amostra de 10% descreve os elegíveis com UF, mas não corrige a ausência de UF nem a seleção de quem faz o ENEM. Usamos os resumos de **todos os elegíveis** para as comparações territoriais principais; a amostra continua nos histogramas e nos filtros de rede/localização.

Este diagnóstico é nacional porque não sabemos a UF escolar de quem não a informa. Atribuir essas pessoas a um estado seria inventar informação.

Há também uma mudança de documentação em 2024: o Inep voltou a disponibilizar código de escola, obtido da matrícula no Censo Escolar, com proteção para escolas com poucos participantes. O [Leia-me de 2024, páginas 7–8](fontes/Leia_Me_Enem_2024.pdf) descreve esse procedimento e a separação entre arquivos de participantes e resultados, sem chave comum. Essa mudança deve acompanhar a interpretação da série; ela não permite quantificar, sozinha, quanto da queda veio da cobertura.

In [ ]:
def mostrar_cobertura_etapa2(area):
    d = cobertura[cobertura.AREA.eq(area)].sort_values('ANO')
    conhecidos = enem_uf[enem_uf.AREA.eq(area)].assign(SOMA=lambda x: x.N * x.MEDIA)
    conhecidos = conhecidos.groupby('ANO').agg(N=('N', 'sum'), SOMA=('SOMA', 'sum'))
    total = enem_nacional[enem_nacional.AREA.eq(area)].set_index('ANO')
    media_com = conhecidos.SOMA / conhecidos.N
    media_sem = (total.MEDIA * total.N - conhecidos.SOMA) / (total.N - conhecidos.N)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(d.ANO, d.COBERTURA_UF_ESCOLA, 'o-', color='#2276A5')
    axs[0].set(title='Cobertura da UF escolar', ylabel='Elegíveis com UF (%)', ylim=(0, 100))
    axs[1].plot(media_com.index, media_com, 'o-', label='UF conhecida')
    axs[1].plot(media_sem.index, media_sem, 's--', label='UF ausente')
    axs[1].set(title='O desempenho dos grupos é parecido?', ylabel='Média (pontos)')
    axs[1].legend(frameon=False)
    for ax in axs:
        ax.set_xticks(ANOS_ESTUDO)
        grade(ax)
    fig.suptitle(area)
    terminar(fig, '01_cobertura')
    print(f'Cobertura em 2020: {d.iloc[0].COBERTURA_UF_ESCOLA:.1f}% | em 2024: {d.iloc[-1].COBERTURA_UF_ESCOLA:.1f}%')
    return d

painel(mostrar_cobertura_etapa2, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'))

## 27. A distribuição sustenta a escolha da mediana? Existem dois picos?

**Pergunta:** o centro resume bem as notas do recorte ou esconde grupos distintos? O histograma usa a amostra por área da etapa 1, agora com contexto da escola. Mude ano, território, rede e largura das faixas. As linhas mostram média e mediana; a faixa sombreada vai de Q1 a Q3.

Nas provas objetivas, a segunda figura desenha três curvas suavizadas sobre as notas positivas. Usamos até 4.000 notas, com semente fixa. Aumentar a largura espalha a contribuição de cada nota por uma vizinhança maior. Isso reduz oscilações, mas também pode esconder dois grupos reais: um pico desaparecer não prova que ele era falso. A quantidade de picos é apenas um diagnóstico exploratório, não um teste conclusivo de bimodalidade. Notas zero permanecem no histograma e nas medidas; são separadas somente nas curvas.

Redação tem notas discretas: seus degraus podem criar picos artificiais numa curva contínua. Nesse caso, mostramos as frequências das notas. A mediana continua sendo o centro por ordenação; **normalidade não é um requisito para usar a média nem um critério automático para escolher a mediana**. Havendo grupos distintos, devemos descrevê-los também.

**Como ler a legenda:** o número é a quantidade de elevações destacadas por um critério de altura relativa, não uma contagem definitiva de grupos de alunos. A largura é um fator multiplicado pelo desvio-padrão das notas da curva; não cria notas novas. O critério exige proeminência de 8% da altura máxima e distância de 15 posições na grade. A escolha desses valores afeta a contagem. Não identificamos perfis de alunos apenas por esse desenho.


In [ ]:
def mostrar_forma(ano, area, territorio, rede, largura):
    d = recortar(amostra, territorio, rede)
    x = d.loc[d.ANO.eq(ano) & d.AREA.eq(area), 'NOTA'].dropna().to_numpy()
    if len(x) < 30:
        print(f'Recorte com {len(x)} notas. Amplie a seleção; não desenhamos curvas para grupos tão pequenos.')
        return
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
    bordas = np.arange(0, max(1000, x.max()) + largura, largura)
    axs[0].hist(x, bins=bordas, weights=np.full(len(x), 100 / len(x)), color='#2276A5', alpha=.8)
    q1, mediana, q3 = np.quantile(x, [.25, .5, .75])
    axs[0].axvspan(q1, q3, color='#E5AA46', alpha=.2, label='Q1 a Q3')
    axs[0].axvline(x.mean(), color='#C95C42', linestyle='--', label=f'Média: {x.mean():.1f}')
    axs[0].axvline(mediana, color='#243747', label=f'Mediana: {mediana:.1f}')
    axs[0].set(xlabel='Nota', ylabel='Notas da amostra (%)', title='Distribuição e medidas centrais')
    axs[0].legend(fontsize=9, frameon=False)
    if area == 'Redação':
        valores, contagens = np.unique(x, return_counts=True)
        axs[1].bar(valores, 100 * contagens / len(x), width=12, color='#298574')
        axs[1].set(title='Frequência de cada nota de Redação', ylabel='Notas da amostra (%)')
    else:
        positivos = x[x > 0]
        if len(positivos) > 4000:
            positivos = np.random.default_rng(42).choice(positivos, 4000, replace=False)
        if len(positivos) > 2 and positivos.std() > 0:
            eixo = np.linspace(*np.quantile(positivos, [.005, .995]), 300)
            for suavizacao in [.15, .25, .35]:
                densidade = gaussian_kde(positivos, bw_method=suavizacao)(eixo)
                picos, _ = find_peaks(densidade, prominence=.08 * densidade.max(), distance=15)
                axs[1].plot(eixo, densidade, label=f'Largura {suavizacao}: {len(picos)} destaque(s)')
            axs[1].legend(fontsize=8, frameon=False)
            print(f'Curvas: {len(positivos):,} notas positivas; contagem depende da largura e do critério de destaque.')
        axs[1].set(title='Os picos persistem ao suavizar?', ylabel='Densidade das notas positivas')
    axs[1].set_xlabel('Nota')
    fig.suptitle(f'{territorio} · {ano} · {area} · {rede}')
    terminar(fig, '02_distribuicao')
    print(f'N da amostra: {len(x):,} | Zero: {100 * (x == 0).mean():.2f}% | Q1–Q3: {q1:.1f} a {q3:.1f}')

painel(mostrar_forma, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
    rede=escolher(REDES_ESTUDO, 'Todas', 'Rede:'),
    largura=escolher([20, 40, 80], 40, 'Faixa:'))

## 28. Como as notas evoluíram em todos os territórios?

**Pergunta:** a queda está em todas as áreas e todos os anos? O painel mostra a trajetória e a mudança em pontos entre anos consecutivos. . A variação de 2020 fica ausente porque 2019 não está no período; isso não equivale a crescimento zero. O intervalo Q1–Q3, disponível nos dados, descreve dispersão e não incerteza da mediana.
As medianas abaixo usam todos os elegíveis com UF escolar. O resumo regional foi calculado reunindo suas notas. A edição de 2020 foi aplicada em 2021: `ANO` é o ano da edição. A série acompanha grupos diferentes, não os mesmos alunos.

In [ ]:
def mostrar_trajetoria(area, nivel, regiao, intervalo, medida):
    d = (enem_regiao if nivel == 'Regiões' else enem_uf).copy()
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    d = d[d.AREA.eq(area)].sort_values([grupo, 'ANO'])
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    d['VARIACAO'] = d.groupby(grupo)[medida].diff()
    d = d[d.ANO.between(*intervalo)]
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
    for nome, g in d.groupby(grupo, observed=True):
        cor = CORES_REGIOES.get(nome) if nivel == 'Regiões' else None
        linha, = axs[0].plot(g.ANO, g[medida], 'o-', label=nome, color=cor, markersize=4)
        axs[1].plot(g.ANO, g.VARIACAO, 'o-', color=linha.get_color(), markersize=4)
    axs[0].set(title=f'{medida.title()} por {nivel.lower()}', ylabel='Nota (pontos)')
    axs[1].set(title='Mudança frente ao ano anterior', ylabel='Diferença em pontos')
    axs[1].axhline(0, color='#667782', linewidth=1)
    for ax in axs:
        ax.set_xticks(range(intervalo[0], intervalo[1] + 1))
        grade(ax)
    fig.legend(*axs[0].get_legend_handles_labels(), loc='lower center', ncol=min(9, d[grupo].nunique()), fontsize=9)
    fig.suptitle(area)
    fig.subplots_adjust(bottom=.24)
    # Reservamos espaço para a legenda quando todas as UFs estão selecionadas.
    fig.tight_layout(rect=(0, .17, 1, .96))
    fig.savefig(PASTA_RESULTADOS / '03_trajetoria.png', bbox_inches='tight')
    plt.show(); plt.close(fig)

painel(mostrar_trajetoria, area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'), intervalo=periodo(),
    medida=escolher(['MEDIANA', 'MEDIA'], 'MEDIANA', 'Medida:'))

## 29. A queda se repete nas cinco áreas?

Um painel único por área pode dar a impressão de que todo o exame se comportou igual. Este mapa de calor compara as cinco áreas, por região ou UF, entre dois anos escolhidos. Azul indica aumento, laranja indica redução. Escolha anos consecutivos para ler a variação anual; períodos maiores mostram a diferença acumulada em pontos, sem atribuí-la a uma causa.

Na apresentação, comece por 2023–2024 e depois volte aos intervalos anteriores. A hipótese “tudo caiu em todo lugar” deve ser conferida célula por célula, não adotada como premissa.

In [ ]:
def mostrar_mudancas(inicio, fim, nivel, regiao):
    if inicio >= fim:
        print('Escolha um ano final posterior ao inicial.')
        return
    d = enem_regiao if nivel == 'Regiões' else enem_uf
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    m = d[d.ANO.isin([inicio, fim])].pivot(index=[grupo, 'AREA'], columns='ANO', values='MEDIANA')
    matriz = (m[fim] - m[inicio]).unstack('AREA').reindex(columns=AREAS_ESTUDO)
    limite = max(1, np.nanmax(abs(matriz.to_numpy())))
    fig, ax = plt.subplots(figsize=(10, max(4, .32 * len(matriz) + 1.5)))
    im = ax.imshow(matriz, cmap='RdBu', vmin=-limite, vmax=limite, aspect='auto')
    ax.set_yticks(range(len(matriz)), matriz.index)
    ax.set_xticks(range(5), ['Linguagens', 'Humanas', 'Natureza', 'Matemática', 'Redação'])
    for i in range(len(matriz)):
        for j in range(5):
            valor = matriz.iloc[i, j]
            ax.text(j, i, f'{valor:+.1f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(valor) > .6 * limite else '#243747')
    ax.set_title(f'Mudança da mediana: {fim} menos {inicio}')
    fig.colorbar(im, ax=ax, label='Pontos', shrink=.8)
    terminar(fig, '04_mudancas_areas')
    salvar(matriz.reset_index(), 'mudancas_areas.csv')

painel(mostrar_mudancas, inicio=escolher(ANOS_ESTUDO, 2023, 'De:'),
    fim=escolher(ANOS_ESTUDO, 2024, 'Até:'), nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'))

## 30. A comparação muda quando separamos redes e localização?

**Pergunta:** as diferenças entre territórios permanecem quando comparamos grupos escolares parecidos? Aqui entram rede e localização urbana/rural. Os filtros permitem qualquer par de UFs ou regiões; Ceará e Sudeste são apenas a seleção inicial.

O cálculo reúne as notas da amostra em cada território, ano e área. Não fazemos uma nova amostragem para cada filtro. Os quartis mostram a dispersão dentro de cada grupo, não a margem de erro da mediana. Grupos com menos de 30 notas não recebem uma curva; o tamanho continua na tabela. O limite é uma precaução de leitura, não garantia de precisão.

Rede privada não significa escola preparatória. Estes arquivos não informam frequência em cursinho nem intensidade da preparação para vestibulares. Uma diferença urbana/rural tampouco mede, sozinha, o efeito de isolamento geográfico.

In [ ]:
import plotly.graph_objects as go
from IPython.display import HTML, display

def mostrar_comparacao(
    area, territorio_a, territorio_b, rede,
    localizacao, intervalo, ver_tabela
):
    fig = go.Figure()
    resumos = []

    for indice, (territorio, cor) in enumerate(zip(
        [territorio_a, territorio_b],
        ['#2276A5', '#C95C42']
    )):
        d = recortar(amostra, territorio, rede)
        d = d[d.AREA.eq(area) & d.ANO.between(*intervalo)]

        if localizacao != 'Todas':
            d = d[d.LOCALIZACAO.eq(localizacao)]

        r = (
            d.groupby('ANO', observed=True).NOTA
            .agg(
                N='size',
                MEDIANA='median',
                Q1=lambda x: x.quantile(.25),
                Q3=lambda x: x.quantile(.75)
            )
            .reset_index()
        )
        r['TERRITORIO'] = territorio
        resumos.append(r)

        r = r[r.N.ge(30)]
        if r.empty:
            continue

        grupo = str(indice)
        rgb = ', '.join(
            str(int(cor[i:i + 2], 16)) for i in (1, 3, 5)
        )

        # Faixa entre o primeiro e o terceiro quartil.
        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.Q1,
            mode='lines',
            line=dict(width=0),
            legendgroup=grupo,
            showlegend=False,
            hoverinfo='skip'
        ))

        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.Q3,
            mode='lines',
            line=dict(width=0),
            fill='tonexty',
            fillcolor=f'rgba({rgb}, 0.10)',
            legendgroup=grupo,
            showlegend=False,
            hoverinfo='skip'
        ))

        # Mediana e valores exibidos ao passar o mouse.
        fig.add_trace(go.Scatter(
            x=r.ANO,
            y=r.MEDIANA,
            mode='lines+markers',
            name=territorio,
            legendgroup=grupo,
            line=dict(color=cor),
            marker=dict(size=8),
            customdata=r[['N', 'Q1', 'Q3']].to_numpy(),
            hovertemplate=(
                '<b>%{fullData.name}</b>'
                '<br>Ano: %{x:.0f}'
                '<br>Mediana: %{y:.2f} pontos'
                '<br>Q1: %{customdata[1]:.2f} pontos'
                '<br>Q3: %{customdata[2]:.2f} pontos'
                '<br>Contagem: %{customdata[0]:.0f}'
                '<extra></extra>'
            )
        ))

    fig.update_layout(
        title=f'{area} · {rede} · localização: {localizacao}',
        yaxis_title='Nota da amostra (pontos)',
        template='plotly_white',
        height=480,
        hovermode='closest',
        xaxis=dict(
            tickmode='array',
            tickvals=list(range(intervalo[0], intervalo[1] + 1))
        ),
        legend_title_text='Território'
    )

    display(HTML(fig.to_html(
        full_html=False,
        include_plotlyjs=True,
        config={'responsive': True, 'displaylogo': False}
    )))

    resultado = pd.concat(resumos, ignore_index=True)
    tabela_opcional(resultado, ver_tabela)
    salvar(resultado, 'comparacao_territorios.csv')


painel(
    mostrar_comparacao,
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    territorio_a=escolher(TERRITORIOS, 'CE', 'Território A:'),
    territorio_b=escolher(TERRITORIOS, 'Sudeste', 'Território B:'),
    rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'),
    localizacao=escolher(
        ['Todas', 'Urbana', 'Rural', 'Não informada'],
        'Todas',
        'Localização:'
    ),
    intervalo=periodo(),
    ver_tabela=widgets.Checkbox(
        value=False,
        description='Ver contagens e quartis'
    )
)



## 31. Juntar as bases sem multiplicar notas

Agora fazemos a junção que sustenta o restante do estudo. São cinco áreas para cada UF e ano no ENEM, mas só uma linha de contexto por UF e ano. Por isso usamos uma ligação **muitos para um**: as cinco linhas recebem o mesmo contexto, sem criar novos participantes.

O código confere duplicatas, preserva as 675 linhas e registra quantas encontraram cada fonte. O ENEM fornece o resultado; Censo e IBGE acrescentam condições do território. Formação docente, jornada e rendimento entram como fontes complementares. A formação adequada é o grupo 1 do indicador do ensino médio, não a adequação específica de cada disciplina do ENEM.

O FUNDEB de dezembro fica identificado como tal. Ele mede uma transferência ao governo estadual em um mês, não gasto anual por aluno, investimento na escola do participante ou execução de uma política. O denominador financeiro é a matrícula estadual de toda a educação básica. Esse dado será explorado separadamente e não entrará automaticamente no conjunto principal de atributos.

A integração principal não inclui mais o recorte mensal do FUNDEB. As validações abaixo conferem a cardinalidade e a correspondência das chaves; ausência de duplicação não demonstra comparabilidade conceitual, que precisa das justificativas das fontes.

In [ ]:
chaves_territorio = ['ANO', 'UF_ESCOLA']
chaves_resultado = ['ANO', 'UF_ESCOLA', 'AREA']
base_integrada = enem_uf.copy()
auditoria_juncoes = []

def juntar_contexto(base, contexto, nome):
    assert not contexto.duplicated(chaves_territorio).any(), f'Chave repetida: {nome}'
    total_antes = len(base)
    nova = base.merge(contexto, on=chaves_territorio, how='left', validate='many_to_one', indicator=True)
    for ano, grupo in nova.groupby('ANO'):
        auditoria_juncoes.append({'FONTE': nome, 'ANO': ano, 'LINHAS_ENEM': len(grupo),
                                  'COM_CORRESPONDENCIA': int(grupo._merge.eq('both').sum())})
    assert len(nova) == total_antes
    assert nova._merge.eq('both').all(), f'Há chaves sem correspondência em {nome}'
    return nova.drop(columns='_merge')

base_integrada = juntar_contexto(base_integrada, censo_uf, 'Censo Escolar')
base_integrada = juntar_contexto(base_integrada, renda_uf, 'IBGE/PNAD')
for nome, tabela in [('Formação docente', formacao), ('Horas-aula', jornada), ('Rendimento', rendimento)]:
    contexto = tabela[tabela.REDE.eq('Total')].drop(columns='REDE')
    base_integrada = juntar_contexto(base_integrada, contexto, nome)

base_integrada = base_integrada.sort_values(chaves_resultado).reset_index(drop=True)
base_integrada['AMPLITUDE_INTERQUARTIL'] = base_integrada.Q3 - base_integrada.Q1
assert len(base_integrada) == 675
assert not base_integrada.duplicated(chaves_resultado).any()
pd.testing.assert_series_equal(base_integrada.N, enem_uf.sort_values(chaves_resultado).reset_index(drop=True).N)
auditoria_juncoes = pd.DataFrame(auditoria_juncoes)
salvar(auditoria_juncoes, 'auditoria_juncoes.csv')
salvar(base_integrada, 'base_territorial_2020_2024.csv')
display(auditoria_juncoes.pivot(index='ANO', columns='FONTE', values='COM_CORRESPONDENCIA'))
print('135 linhas por ano: 27 UFs × 5 áreas. A junção preservou as 675 linhas e as contagens do ENEM.')

FONTE,Censo Escolar,Formação docente,Horas-aula,IBGE/PNAD,Rendimento
ANO,,,,,
2020,135,135,135,135,135
2021,135,135,135,135,135
2022,135,135,135,135,135
2023,135,135,135,135,135
2024,135,135,135,135,135


135 linhas por ano: 27 UFs × 5 áreas. A junção preservou as 675 linhas e as contagens do ENEM.


## 32. Onde estão as carências na oferta escolar?

**Pergunta:** os recursos estão igualmente disponíveis nos territórios e nas redes? A cor representa disponibilidade; não é uma nota de qualidade escolar. A tabela opcional informa quantas escolas responderam cada item.

O universo é o das escolas ativas com ensino médio no Censo, não apenas as escolas dos participantes do ENEM. Cada escola tem o mesmo peso neste gráfico. Para regiões, reunimos as escolas dos estados antes de calcular o percentual. Valores ausentes não viram “não”; o denominador é o número de respostas válidas do item. Água, internet ou biblioteca não informam, por si só, qualidade de uso, manutenção ou acesso efetivo.

In [ ]:
RECURSOS = {'IN_AGUA_POTAVEL': 'Água potável', 'IN_ENERGIA_REDE_PUBLICA': 'Energia da rede',
    'IN_ESGOTO_REDE_PUBLICA': 'Esgoto da rede', 'IN_REFEITORIO': 'Refeitório',
    'IN_SALA_DIRETORIA': 'Sala da direção', 'IN_SALA_ATENDIMENTO_ESPECIAL': 'Atendimento especial',
    'IN_COMPUTADOR': 'Computador', 'IN_ALIMENTACAO': 'Alimentação',
    'IN_INTERNET_ALUNOS': 'Internet para alunos', 'IN_BIBLIOTECA_SALA_LEITURA': 'Biblioteca / leitura',
    'IN_LABORATORIO_CIENCIAS': 'Laboratório de ciências', 'IN_LABORATORIO_INFORMATICA': 'Laboratório de informática',
    'IN_QUADRA_ESPORTES': 'Quadra de esportes', 'IN_BANHEIRO_PNE': 'Banheiro acessível'}

def mostrar_infraestrutura(ano, nivel, regiao, rede, localizacao, ver_tabela):
    d = recortar(escolas[escolas.ANO.eq(ano)], regiao, rede)
    if localizacao != 'Todas':
        d = d[d.LOCALIZACAO.eq(localizacao)]
    if d.empty:
        print('Não há escolas neste recorte.')
        return
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    m = d.groupby(grupo, observed=True)[list(RECURSOS)].mean() * 100
    contagens = d.groupby(grupo, observed=True)[list(RECURSOS)].count()
    fig, ax = plt.subplots(figsize=(13, max(4.5, .32 * len(m) + 2.2)))
    im = ax.imshow(m, vmin=0, vmax=100, cmap='YlGnBu', aspect='auto')
    ax.set_xticks(range(len(RECURSOS)), RECURSOS.values(), rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(len(m)), m.index)
    if len(m) <= 7:
        for i in range(len(m)):
            for j in range(len(RECURSOS)):
                v = m.iloc[i, j]
                ax.text(j, i, f'{v:.0f}' if pd.notna(v) else '—', ha='center', va='center', fontsize=9,
                        color='white' if v > 65 else '#243747')
    ax.set_title(f'Escolas de ensino médio com o recurso (%) · {ano} · {rede} · {localizacao}')
    fig.colorbar(im, ax=ax, label='Respostas válidas com recurso (%)', shrink=.7)
    terminar(fig, '06_infraestrutura')
    print(f'{len(d):,} escolas no recorte. Percentuais calculados por item, entre respostas válidas.')
    tabela_opcional(contagens.rename(columns=RECURSOS).reset_index(), ver_tabela)

painel(mostrar_infraestrutura, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Brasil (UF informada)'] + REGIOES_ESTUDO, 'Brasil (UF informada)', 'Região:'),
    rede=escolher(['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada'], 'Públicas', 'Rede:'),
    localizacao=escolher(['Todas', 'Urbana', 'Rural'], 'Todas', 'Localização:'),
    ver_tabela=widgets.Checkbox(value=False, description='Ver número de respostas'))

## 33. A oferta escolar mudou ao longo dos cinco anos?

Ver a infraestrutura em apenas um ano não permite acompanhar sua evolução. Aqui seguimos um recurso ou o percentual de matrículas em tempo integral entre 2020 e 2024, com filtros de rede, região e localização. O tempo integral usa matrículas como peso; os demais recursos usam escolas com resposta válida. A diferença de denominadores fica indicada no eixo.

O crescimento da oferta não demonstra sozinho um efeito nas notas. O Censo registra a situação da escola na referência anual; não sabemos quanto tempo cada participante esteve exposto ao recurso. Além disso, o programa federal Escola em Tempo Integral foi criado em 2023, enquanto iniciativas estaduais e escolas de tempo integral já existiam. Não chamamos todo aumento de adesão a esse programa.

In [ ]:
def mostrar_oferta(indicador, nivel, regiao, rede, localizacao, intervalo):
    d = recortar(escolas, regiao, rede)
    d = d[d.ANO.between(*intervalo)]
    if localizacao != 'Todas':
        d = d[d.LOCALIZACAO.eq(localizacao)]
    grupo = 'REGIAO' if nivel == 'Regiões' else 'UF_ESCOLA'
    if indicador == 'Tempo integral':
        r = d.groupby(['ANO', grupo], observed=True)[['QT_MAT_MED', 'QT_MAT_MED_INT']].sum(min_count=1)
        r['VALOR'] = 100 * r.QT_MAT_MED_INT / r.QT_MAT_MED.replace(0, np.nan)
        eixo = 'Matrículas do ensino médio em tempo integral (%)'
    else:
        coluna = next(c for c, nome in RECURSOS.items() if nome == indicador)
        r = (100 * d.groupby(['ANO', grupo], observed=True)[coluna].mean()).to_frame('VALOR')
        eixo = 'Escolas com o recurso, entre respostas válidas (%)'
    r = r.reset_index()
    fig, ax = plt.subplots(figsize=(11, 5.5))
    for nome, g in r.groupby(grupo, observed=True):
        ax.plot(g.ANO, g.VALOR, 'o-', label=nome, markersize=4,
                color=CORES_REGIOES.get(nome) if nivel == 'Regiões' else None)
    ax.set(title=f'{indicador} · {rede} · {localizacao}', ylabel=eixo, ylim=(0, 100))
    ax.set_xticks(range(intervalo[0], intervalo[1] + 1)); grade(ax)
    if len(r):
        ax.legend(frameon=False, ncol=min(9, r[grupo].nunique()), fontsize=9, loc='upper center', bbox_to_anchor=(.5, -.12))
    terminar(fig, '07_oferta_ao_longo_do_tempo')
    salvar(r, 'evolucao_oferta.csv')

painel(mostrar_oferta, indicador=escolher(['Tempo integral'] + list(RECURSOS.values()), 'Tempo integral', 'Indicador:'),
    nivel=escolher(['Regiões', 'UFs'], 'Regiões', 'Comparar:'),
    regiao=escolher(['Brasil (UF informada)'] + REGIOES_ESTUDO, 'Brasil (UF informada)', 'Região:'),
    rede=escolher(['Todas', 'Públicas', 'Federal', 'Estadual', 'Municipal', 'Privada'], 'Públicas', 'Rede:'),
    localizacao=escolher(['Todas', 'Urbana', 'Rural'], 'Todas', 'Localização:'), intervalo=periodo())

## 34. Criar um recorte coerente para a rede estadual

A integração geral descreve todas as redes. Para estudar uma unidade próxima da atuação das secretarias estaduais, usamos notas, infraestrutura e indicadores da **rede estadual**. A renda continua descrevendo toda a UF. Essa delimitação vem da pergunta de gestão e da compatibilidade das fontes; não foi escolhida para obter uma correlação maior.

Recalculamos mediana, média e quartis diretamente de todas as notas elegíveis estaduais de 2020–2024. Não juntamos medianas urbanas e rurais. A amostra é mantida para comparar a diferença causada pelo sorteio e explorar distribuições; ela não define mais o alvo estadual.

As contagens são confrontadas com os resumos completos anteriores. Também conferimos a média pela soma ponderada dos grupos, operação válida para a média, mas não para a mediana. As bases geral e estadual compartilham participantes e não devem ser empilhadas como exemplos independentes.


In [ ]:
chaves_territorio = ['ANO', 'UF_ESCOLA']
chaves_resultado = ['ANO', 'UF_ESCOLA', 'AREA']

amostra_estadual = amostra[amostra.REDE.eq('Estadual')]
resumo_amostra = amostra_estadual.groupby(chaves_resultado, observed=True).NOTA.agg(
    N_AMOSTRA='size', MEDIANA_AMOSTRA='median').reset_index()
base_estadual = enem_estadual_completo.merge(resumo_amostra, on=chaves_resultado, validate='one_to_one')
grupos = enem_redes[enem_redes.REDE.eq('Estadual')].copy()
grupos['SOMA_NOTAS'] = grupos.N * grupos.MEDIA
conferencia = grupos.groupby(chaves_resultado, as_index=False).agg(N_ANTERIOR=('N', 'sum'), SOMA_NOTAS=('SOMA_NOTAS', 'sum'))
conferencia = base_estadual.merge(conferencia, on=chaves_resultado, validate='one_to_one')
assert conferencia.N.eq(conferencia.N_ANTERIOR).all()
assert np.allclose(conferencia.MEDIA, conferencia.SOMA_NOTAS / conferencia.N_ANTERIOR)
censo_rede = ler('censo_uf_rede_2020_2024.csv')
for contexto in [censo_rede[censo_rede.REDE.eq('Estadual')].drop(columns='REDE'), renda_uf,
                 formacao[formacao.REDE.eq('Estadual')].drop(columns='REDE'),
                 jornada[jornada.REDE.eq('Estadual')].drop(columns='REDE'),
                 rendimento[rendimento.REDE.eq('Estadual')].drop(columns='REDE')]:
    base_estadual = base_estadual.merge(contexto, on=chaves_territorio, how='left',
        validate='many_to_one', indicator=True)
    assert base_estadual._merge.eq('both').all()
    base_estadual = base_estadual.drop(columns='_merge')
base_estadual['REDE'] = 'Estadual'
base_estadual['AMPLITUDE_INTERQUARTIL'] = base_estadual.Q3 - base_estadual.Q1
assert len(base_estadual) == 675 and not base_estadual.duplicated(chaves_resultado).any()
salvar(base_estadual, 'base_rede_estadual_2020_2024.csv')
print('Rede estadual: 675 resultados completos. Menor número de elegíveis:', base_estadual.N.min())

Rede estadual: 675 resultados completos. Menor número de elegíveis: 816



## 35. Renda, condições escolares e desempenho aparecem associados?

**Pergunta:** UFs com determinado indicador mais alto também tendem a ter maior desempenho? Cada ponto é uma UF, dentro de um mesmo ano e área. Os filtros alternam todas as redes e o recorte estadual. Cor identifica região; a correlação de Spearman resume se as posições dos estados nos dois eixos tendem a caminhar juntas.

O painel examina uma característica de cada vez. Uma associação entre renda e nota não demonstra um efeito individual da renda; uma associação com laboratório pode refletir diferenças de renda, rede, participação e outras condições. Não há ajuste estatístico nem teste causal aqui. Não usamos dezenas de p-valores para declarar “fatores comprovados”.

O gráfico à direita compara os anos para ver se a relação persiste. A formação adequada, o tempo integral, a jornada e o abandono representam dimensões diferentes da oferta. Aprovação, reprovação e abandono são relacionados entre si; não devem entrar todos automaticamente num modelo.

**Revisão:** Corrigir rótulos após recalcular o alvo.

In [ ]:
ATRIBUTOS = {
    'Renda por pessoa (R$ de 2024)': 'RENDA_PC_REAIS_2024_IPCA',
    'Matrículas em tempo integral (%)': 'PCT_MATRICULAS_INTEGRAL_EM',
    'Docentes com formação adequada (%)': 'PCT_DOCENTES_ADEQUADOS_EM',
    'Horas-aula por dia': 'HORAS_AULA_DIA_EM',
    'Abandono no ensino médio (%)': 'PCT_ABANDONO_EM',
    'Escolas com internet para alunos (%)': 'PCT_IN_INTERNET_ALUNOS',
    'Escolas com laboratório de ciências (%)': 'PCT_IN_LABORATORIO_CIENCIAS',
    'Escolas com biblioteca/leitura (%)': 'PCT_IN_BIBLIOTECA_SALA_LEITURA',
    'Escolas com água potável (%)': 'PCT_IN_AGUA_POTAVEL',
    'Escolas rurais (%)': 'PCT_ESCOLAS_RURAIS_EM'}

def correlacao_segura(d, coluna, alvo='MEDIANA'):
    pares = d[[coluna, alvo]].dropna()
    if len(pares) < 4 or pares[coluna].nunique() < 2 or pares[alvo].nunique() < 2:
        return np.nan
    return spearmanr(pares[coluna], pares[alvo]).statistic

def mostrar_associacao(ano, area, atributo, regiao, universo, medida, ver_tabela):
    origem = base_integrada if universo == 'Todas as redes' else base_estadual
    d = origem[origem.AREA.eq(area)]
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    coluna = ATRIBUTOS[atributo]
    atual = d[d.ANO.eq(ano)].dropna(subset=[coluna, medida])
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1.5, 1]})
    for nome, g in atual.groupby('REGIAO', observed=True):
        axs[0].scatter(g[coluna], g[medida], label=nome, color=CORES_REGIOES[nome], s=45)
        for _, linha in g.iterrows():
            axs[0].annotate(linha.UF_ESCOLA, (linha[coluna], linha[medida]), xytext=(4, 3), textcoords='offset points', fontsize=8)
    rho = correlacao_segura(atual, coluna, medida)
    axs[0].set(xlabel=atributo, ylabel=f'{medida.title()} (pontos)', title=f'{ano} · {len(atual)} UFs · Spearman = {rho:.2f}')
    axs[0].legend(frameon=False, fontsize=8)
    historico = pd.DataFrame([{'ANO': a, 'SPEARMAN': correlacao_segura(g, coluna, medida), 'N_UFS': len(g)}
                             for a, g in d.groupby('ANO')])
    axs[1].plot(historico.ANO, historico.SPEARMAN, 'o-', color='#298574')
    axs[1].axhline(0, color='#667782', linewidth=1)
    axs[1].set(title='A associação se mantém?', ylabel='Correlação de Spearman', ylim=(-1.05, 1.05), xticks=ANOS_ESTUDO)
    for ax in axs: grade(ax)
    fig.suptitle(f'{area} · {universo}')
    terminar(fig, '08_associacoes')
    print('Ambos os universos usam resumos completos dos elegíveis; somente a rede e o contexto escolar mudam.')
    print('Correlação exploratória; grupos com menos de 4 UFs ou sem variação ficam sem coeficiente.')
    tabela_opcional(atual[['UF_ESCOLA', coluna, medida, 'N']], ver_tabela)

painel(mostrar_associacao, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'),
    atributo=escolher(list(ATRIBUTOS), 'Renda por pessoa (R$ de 2024)', 'Indicador:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'),
    medida=escolher(['MEDIANA', 'MEDIA'], 'MEDIANA', 'Medida:'),
    ver_tabela=widgets.Checkbox(value=False, description='Ver valores das UFs'))



## 36. A associação muda conforme a área e o ano?

O mapa de calor reúne as cinco áreas sem somar suas notas. Cada célula é uma correlação entre UFs, para o ano e universo selecionados. A cor representa associação, **não importância de variável, porcentagem explicada ou efeito de política pública**. Os resultados completos de todos os anos serão salvos, incluindo quantas UFs participaram de cada cálculo.

Se uma relação aparecer em apenas um ano, merece cautela. Se duas características se movem juntas, um futuro modelo pode ter dificuldade em separar suas contribuições. Por isso a seleção de atributos usará justificativa, disponibilidade e redundância, além destes gráficos.

**Revisão:** Recalcular associações com o alvo estadual completo.

In [ ]:
associacoes = []
for universo, origem in [('Todas as redes', base_integrada), ('Rede estadual', base_estadual)]:
    for (ano, area), grupo in origem.groupby(['ANO', 'AREA'], observed=True):
        for nome, coluna in ATRIBUTOS.items():
            pares = grupo[[coluna, 'MEDIANA']].dropna()
            associacoes.append({'UNIVERSO': universo, 'ANO': ano, 'AREA': area, 'ATRIBUTO': nome,
                'SPEARMAN': correlacao_segura(grupo, coluna), 'N_UFS': len(pares)})
associacoes = pd.DataFrame(associacoes)
salvar(associacoes, 'associacoes_por_ano_area.csv')

def mostrar_matriz_associacoes(ano, universo):
    d = associacoes[associacoes.ANO.eq(ano) & associacoes.UNIVERSO.eq(universo)]
    m = d.pivot(index='ATRIBUTO', columns='AREA', values='SPEARMAN').reindex(index=ATRIBUTOS, columns=AREAS_ESTUDO)
    fig, ax = plt.subplots(figsize=(11, 6))
    im = ax.imshow(m, cmap='RdBu', vmin=-1, vmax=1, aspect='auto')
    ax.set_yticks(range(len(m)), m.index, fontsize=9)
    ax.set_xticks(range(5), ['Linguagens', 'Humanas', 'Natureza', 'Matemática', 'Redação'])
    for i in range(len(m)):
        for j in range(5):
            v = m.iloc[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', color='white' if abs(v) > .65 else '#243747')
    ax.set_title(f'Associações entre UFs · {ano} · {universo}')
    fig.colorbar(im, ax=ax, label='Spearman', shrink=.8)
    terminar(fig, '09_associacoes_areas')

painel(mostrar_matriz_associacoes, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'))



## 38. Quanto a composição dos participantes ajuda a entender a mudança?

**Pergunta:** parte da variação aparece porque a proporção de redes e localizações mudou? À esquerda, mostramos a composição por rede. À direita, recalculamos a média do ano final usando a distribuição do ano inicial entre grupos **UF × rede × localização**. Entram grupos com pelo menos 30 notas completas nos dois anos.

A barra “com composição inicial” mantém os pesos dos grupos, mas permite que suas médias mudem. A diferença entre ela e a variação observada indica sensibilidade à composição. É uma decomposição descritiva da **média**, não da mediana. O percentual de notas cobertas pelos grupos comuns informa quanto do recorte foi efetivamente comparado. As categorias não informadas permanecem como grupos próprios.

Isso não elimina diferenças de renda ou seleção dentro de cada grupo e não separa efeitos de pandemia, currículo ou políticas. O objetivo é testar uma explicação verificável antes de atribuir a queda a um evento.

A parcela de composição depende de usar o ano inicial como referência. Outra decomposição pode distribuir as parcelas de outro modo. Ela não acompanha as mesmas pessoas e não deve ser apresentada como percentual causal da queda.

In [ ]:

def decompor_composicao(territorio, area, inicio, fim):
    d = recortar(enem_redes, territorio)
    d = d[d.AREA.eq(area)]
    chave = ['UF_ESCOLA', 'REDE', 'LOCALIZACAO']
    a = d[d.ANO.eq(inicio)].set_index(chave)
    b = d[d.ANO.eq(fim)].set_index(chave)
    comum = a.index.intersection(b.index)
    comum = comum[(a.loc[comum, 'N'].ge(30) & b.loc[comum, 'N'].ge(30)).to_numpy()]
    if len(comum) == 0:
        return None
    ga, gb = a.loc[comum], b.loc[comum]
    wa, wb = ga.N / ga.N.sum(), gb.N / gb.N.sum()
    media_a, media_b = (wa * ga.MEDIA).sum(), (wb * gb.MEDIA).sum()
    media_fixa = (wa * gb.MEDIA).sum()
    return {'OBSERVADA': media_b - media_a, 'COMPOSICAO_INICIAL': media_fixa - media_a,
        'PARCELA_COMPOSICAO': media_b - media_fixa,
        'COBERTURA_INICIAL': 100 * ga.N.sum() / a.N.sum(),
        'COBERTURA_FINAL': 100 * gb.N.sum() / b.N.sum(), 'N_GRUPOS': len(comum)}

def mostrar_composicao(territorio, area, inicio, fim):
    if inicio >= fim:
        print('Escolha um ano final posterior ao inicial.')
        return
    resultado = decompor_composicao(territorio, area, inicio, fim)
    if resultado is None:
        print('Não há grupos comuns com 30 notas em ambos os anos.')
        return
    d = recortar(enem_redes, territorio)
    d = d[d.AREA.eq(area) & d.ANO.isin([inicio, fim])]
    m = d.groupby(['ANO', 'REDE']).N.sum().unstack(fill_value=0)
    m = m.div(m.sum(axis=1), axis=0) * 100
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
    m.plot.barh(stacked=True, ax=axs[0], colormap='tab10', width=.5)
    axs[0].set(xlabel='Notas elegíveis (%)', ylabel='', xlim=(0, 100), title='Participação de cada rede')
    axs[0].legend(frameon=False, fontsize=8, loc='upper center', bbox_to_anchor=(.5, -.17), ncol=3)
    valores = [resultado[c] for c in ['OBSERVADA', 'COMPOSICAO_INICIAL', 'PARCELA_COMPOSICAO']]
    barras = axs[1].barh(['Observada', 'Com composição inicial', 'Parcela de composição'], valores,
                       color=['#243747', '#2276A5', '#C95C42'])
    axs[1].bar_label(barras, fmt='%+.1f', padding=4)
    axs[1].axvline(0, color='#9CA9B1', linewidth=1)
    axs[1].margins(x=.25)
    axs[1].set(xlabel='Mudança da média (pontos)', title='Grupos comuns aos dois anos')
    fig.suptitle(f'{territorio} · {area} · {inicio} a {fim}')
    terminar(fig, '11_composicao')
    print(f'Cobertura dos grupos comuns: {resultado["COBERTURA_INICIAL"]:.1f}% em {inicio}; '
          f'{resultado["COBERTURA_FINAL"]:.1f}% em {fim}. Grupos: {resultado["N_GRUPOS"]}.')

painel(mostrar_composicao, territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
    area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'), inicio=escolher(ANOS_ESTUDO, 2023, 'De:'),
    fim=escolher(ANOS_ESTUDO, 2024, 'Até:'))

## 39. Há perfis diferentes de desempenho entre as áreas?

**Pergunta:** um mesmo participante ocupa posições diferentes nas áreas? Para responder, precisamos das quatro notas na mesma linha original. A amostra por área da primeira etapa não serve para isso, porque os sorteios das áreas foram separados. Este painel usa uma amostra conjunta de 10%, por UF e ano, entre quem tem as quatro provas objetivas válidas e UF conhecida, com semente fixa.

Transformamos cada nota em posição percentual dentro da área e da edição nessa amostra conjunta. Comparamos a média das posições em MT/CN com a média em LC/CH. Uma diferença de pelo menos 15 pontos percentuais é uma regra descritiva ajustável, não um diagnóstico de aptidão. Não permite concluir qual curso o aluno procura nem por que prefere uma área.

Os filtros mostram a composição desses perfis em todas as UFs, regiões e anos. Não seguimos a mesma pessoa entre edições. Renda familiar ligada às notas está disponível no material processado até 2023; em 2024 não tentamos ligar os arquivos por posição de linha. Por esse motivo ela não é um atributo individual completo para os cinco anos.

In [ ]:
def mostrar_perfis(ano, territorio, rede, limite):
    d = recortar(perfis, territorio, rede)
    d = d[d.ANO.eq(ano)].copy()
    if len(d) < 30:
        print(f'Recorte com {len(d)} participantes. Amplie os filtros.')
        return
    d['POSICAO_MT_CN'] = d[['PERCENTIL_MT', 'PERCENTIL_CN']].mean(axis=1)
    d['POSICAO_LC_CH'] = d[['PERCENTIL_LC', 'PERCENTIL_CH']].mean(axis=1)
    diferenca = d.POSICAO_MT_CN - d.POSICAO_LC_CH
    ordem = ['Maior posição em MT/CN', 'Posições próximas', 'Maior posição em LC/CH']
    perfil = np.select([diferenca.ge(limite), diferenca.le(-limite)], [ordem[0], ordem[2]], default=ordem[1])
    proporcao = pd.Series(perfil).value_counts(normalize=True).reindex(ordem, fill_value=0) * 100
    pontos = d.sample(min(2500, len(d)), random_state=42)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].scatter(pontos.POSICAO_LC_CH, pontos.POSICAO_MT_CN, s=8, alpha=.12, color='#2276A5')
    axs[0].plot([0, 100], [0, 100], '--', color='#667782')
    axs[0].set(xlabel='Posição média em LC/CH', ylabel='Posição média em MT/CN', xlim=(0, 100), ylim=(0, 100), title='Mesma pessoa, quatro áreas')
    barras = axs[1].barh(ordem, proporcao, color=['#2276A5', '#87969F', '#C95C42'])
    axs[1].bar_label(barras, fmt='%.1f%%', padding=4)
    axs[1].set(xlim=(0, 100), xlabel='Participantes do recorte (%)', title=f'Diferença de pelo menos {limite} p.p.')
    fig.suptitle(f'{territorio} · {ano} · {rede}')
    terminar(fig, '12_perfis')
    print(f'{len(d):,} participantes com quatro provas. O gráfico de pontos mostra até 2.500; as barras usam todos do recorte.')

painel(mostrar_perfis, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    territorio=escolher(TERRITORIOS, 'Brasil (UF informada)', 'Território:'),
    rede=escolher(REDES_ESTUDO, 'Estadual', 'Rede:'), limite=escolher([10, 15, 20], 15, 'Diferença:'))



## 40. Uma variável disponível pode não ser adequada à pergunta

Aumentar a quantidade de fontes não garante uma base melhor. O FUNDEB disponível neste pacote descreve dezembro, não o investimento anual no ensino médio estadual. Dividir esse valor pelas matrículas não corrige a incompatibilidade. Retiramos esse indicador dos gráficos de associação, da junção principal e das entradas de previsão.

Uma ampliação futura precisa de gasto executado anual, rede e etapa compatíveis, correção monetária e data de divulgação conhecida. O mesmo cuidado vale para renda individual e indicadores de aprendizagem de outras edições. A tabela abaixo registra exclusões para que elas não pareçam esquecimento.


In [ ]:

exclusoes = pd.DataFrame([
    ['FUNDEB apenas em dezembro', 'Mês isolado não mede investimento anual ou gasto na escola', 'Buscar execução anual com rede e etapa compatíveis'],
    ['Renda individual entre versões incompatíveis do ENEM', 'Não substituir renda familiar por renda média da UF', 'Auditar questionários e acesso antes de integrar'],
    ['Notas e perfis do próprio ano-alvo', 'Contêm a resposta que se deseja prever', 'Usar apenas histórico disponível antes da previsão'],
    ['Escolher uma UF por ter maior correlação', 'Favorece seleção oportunista e reduz a série', 'Examinar cobertura, comparabilidade e necessidade pública']],
    columns=['Informação', 'Por que não entra agora', 'Condição para reconsiderar'])
display(exclusoes.style.hide(axis='index').set_properties(**{'text-align': 'left'}))
salvar(exclusoes, 'exclusoes_justificadas.csv')


Informação,Por que não entra agora,Condição para reconsiderar
FUNDEB apenas em dezembro,Mês isolado não mede investimento anual ou gasto na escola,Buscar execução anual com rede e etapa compatíveis
Renda individual entre versões incompatíveis do ENEM,Não substituir renda familiar por renda média da UF,Auditar questionários e acesso antes de integrar
Notas e perfis do próprio ano-alvo,Contêm a resposta que se deseja prever,Usar apenas histórico disponível antes da previsão
Escolher uma UF por ter maior correlação,Favorece seleção oportunista e reduz a série,"Examinar cobertura, comparabilidade e necessidade pública"


## 41. Pandemia, políticas e TRI: o que podemos afirmar?

Os resultados precisam ser confrontados com a documentação do período. Os eventos abaixo são **contexto histórico**, não colunas de tratamento atribuídas aos participantes.

| Referência | O que está documentado | Como entra na leitura |
|---|---|---|
| 2020–2021: interrupção das aulas presenciais | O Inep documentou ampla suspensão e alterações no calendário escolar. [Nota informativa do Ideb 2021](https://download.inep.gov.br/educacao_basica/portal_ideb/planilhas_para_download/2021/nota_informativa_ideb_2021.pdf) | Torna plausível investigar consequências posteriores; não identifica quanto cada participante perdeu nem um atraso específico até 2024. |
| 2023: criação do Programa Escola em Tempo Integral | A política federal prevê apoio à ampliação de matrículas. [Notícia de 31/07/2023](https://www.gov.br/planalto/pt-br/acompanhe-o-planalto/noticias/2023/07/escola-de-qualidade-e-base-para-sociedade-menos-desigual-diz-lula) | Motiva observar oferta e jornada. Não confundimos anúncio com execução nem toda matrícula integral com adesão ao novo programa. |
| 2024: apoio ao acesso ao ENEM no Ceará | Houve divulgação de transporte gratuito em metrô e VLT nos dias de prova. [Governo do Ceará, 31/10/2024](https://www.ce.gov.br/2024/10/31/inscritos-no-enem-contarao-com-metro-e-vlts-gratuitamente-nos-dois-domingos-de-prova/) | Pode motivar estudos de participação. Não sabemos quem utilizou o transporte; a notícia não demonstra aumento de nota. |
| ENEM: notas e comparabilidade | A TRI considera características dos itens e padrão de respostas; Redação segue critérios próprios. [Guia do participante sobre as notas](https://download.inep.gov.br/educacao_basica/enem/guia_participante/2013/guia_do_participante_notas.pdf) | Quantidades iguais de acertos não exigem notas iguais. Uma diferença entre anos não comprova mudança do método de correção. |

**Conclusão sobre a pandemia:** nossos dados não permitem atribuir a queda de 2024 a um “lag da pandemia”. Não temos acompanhamento individual, exposição escolar detalhada nem um grupo contrafactual adequado. Cobertura, composição e resultados diferem entre edições. Criar uma coluna `pandemia = 1` para todo mundo não resolveria isso.

**Conclusão sobre escolas preparatórias:** não há uma variável que identifique essa atividade de forma comparável em todas as UFs e anos. A hipótese fica pendente; não usamos rede privada como substituto.

O próximo gráfico é um diagnóstico complementar nacional de **2023 e 2024**, anos cujas respostas e gabaritos já foram conferidos nesta preparação. Essa limitação está restrita ao anexo de acertos; a base territorial e os demais painéis cobrem 2020–2024. Não ampliamos a conclusão sobre TRI a anos ou territórios que não foram processados nesse diagnóstico.

In [ ]:
acertos = ler('notas_por_numero_acertos_2023_2024.csv')
CODIGOS_AREAS = {'Linguagens': 'LC', 'Ciências Humanas': 'CH', 'Ciências da Natureza': 'CN', 'Matemática': 'MT'}

def mostrar_acertos(area):
    d = acertos[acertos.AREA.eq(CODIGOS_AREAS[area])]
    fig, ax = plt.subplots(figsize=(10, 4.8))
    for ano, g in d.groupby('ANO'):
        g = g[g.N.ge(100)].sort_values('ACERTOS')
        ax.plot(g.ACERTOS, g.NOTA_MEDIA, 'o-', label=str(ano), markersize=3)
    ax.set(title=f'{area}: média da nota por quantidade de acertos', xlabel='Acertos', ylabel='Nota média')
    ax.legend(frameon=False); grade(ax)
    terminar(fig, '14_acertos')
    print('Elegíveis nacionais, inclusive sem UF. Grupos com pelo menos 100 notas. Mesmos acertos não significam mesmas questões acertadas.')

painel(mostrar_acertos, area=escolher(list(CODIGOS_AREAS), 'Matemática', 'Área:'))



> Adicionar aspas



## 42. Transformar resultados em perguntas úteis para a gestão

**Pergunta:** em quais UFs convivem menor desempenho observado, menor renda e menor disponibilidade de um recurso? O painel usa a rede estadual para aproximar o recorte da atuação das secretarias. Renda permanece como contexto da população da UF. Tamanho do ponto indica a proporção de escolas estaduais sem o recurso, entre respostas válidas.

As linhas tracejadas são medianas entre UFs, usadas apenas para orientar a leitura; não são metas oficiais nem um critério de elegibilidade. As UFs abaixo delas não recebem automaticamente uma classificação de prioridade. Primeiro precisamos conferir cobertura, desigualdades internas, demanda local e evidências de outras avaliações, como o Saeb.

Um território com notas baixas e muitas carências pode motivar uma visita técnica. Um com infraestrutura mais disponível e notas baixas pode motivar investigação de outras dificuldades. Um com resultados altos em contexto desfavorável pode ser selecionado para estudo de práticas. Nenhuma dessas situações, sozinha, demonstra qual intervenção funcionará.

**Revisão:** Corrigir rótulo do diagnóstico territorial.

In [ ]:

def mostrar_diagnostico(ano, area, recurso, regiao):
    d = base_estadual[base_estadual.ANO.eq(ano) & base_estadual.AREA.eq(area)].copy()
    if regiao != 'Todas':
        d = d[d.REGIAO.eq(regiao)]
    coluna = 'PCT_' + next(c for c, nome in RECURSOS.items() if nome == recurso)
    d['PCT_SEM_RECURSO'] = 100 - d[coluna]
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for nome, g in d.groupby('REGIAO', observed=True):
        ax.scatter(g.RENDA_PC_REAIS_2024_IPCA, g.MEDIANA, s=40 + 3 * g.PCT_SEM_RECURSO,
            color=CORES_REGIOES[nome], alpha=.65, label=nome, edgecolors='white')
        for _, linha in g.iterrows():
            ax.annotate(linha.UF_ESCOLA, (linha.RENDA_PC_REAIS_2024_IPCA, linha.MEDIANA),
                        xytext=(5, 4), textcoords='offset points', fontsize=9)
    ax.axvline(d.RENDA_PC_REAIS_2024_IPCA.median(), linestyle='--', color='#A2AEB7')
    ax.axhline(d.MEDIANA.median(), linestyle='--', color='#A2AEB7')
    ax.set(title=f'Rede estadual · {ano} · {area}', xlabel='Renda por pessoa na UF (R$ de 2024)', ylabel='Mediana dos elegíveis (pontos)')
    ax.legend(frameon=False, fontsize=9); grade(ax)
    terminar(fig, '15_diagnostico_territorial')
    print(f'Pontos maiores: maior percentual de escolas sem {recurso.lower()}. Não é um ranking de prioridade.')
    salvar(d[['ANO', 'UF_ESCOLA', 'AREA', 'N', 'N_AMOSTRA', 'MEDIANA',
              'RENDA_PC_REAIS_2024_IPCA', 'PCT_SEM_RECURSO']], 'diagnostico_territorial.csv')

painel(mostrar_diagnostico, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    area=escolher(AREAS_ESTUDO, 'Ciências da Natureza', 'Área:'),
    recurso=escolher(list(RECURSOS.values()), 'Laboratório de ciências', 'Recurso:'),
    regiao=escolher(['Todas'] + REGIOES_ESTUDO, 'Todas', 'Região:'))

# AQUI TEVE MELHORIA

## 43. Quais atributos fazem sentido para a etapa 3?

A escolha começa pela pergunta, não pela quantidade de colunas. Pretendemos avaliar se condições observadas ajudam a estimar o desempenho futuro e a selecionar territórios para investigação. Uma primeira lista reúne renda, oferta de tempo integral, formação docente, abandono, internet e laboratório de ciências. O desempenho anterior será uma referência indispensável.

| Informação | Papel | Decisão inicial |
|---|---|---|
| Mediana por área | Resultado a estimar | Manter o mesmo alvo entre UFs; média como análise de sensibilidade |
| Mediana do ano anterior | Histórico e referência simples | Comparar o futuro modelo com “repetir a nota anterior” |
| Renda real por pessoa | Contexto econômico | Manter uma versão de preços; não incluir simultaneamente renda nominal e deflacionada |
| Matrículas integrais | Oferta de jornada ampliada | Candidata; não identifica participação individual em programa |
| Formação docente adequada | Condição de ensino | Candidata; indicador agregado do ensino médio |
| Abandono | Permanência escolar | Candidata; revisar mudanças administrativas durante a pandemia |
| Internet e laboratório de ciências | Dimensões específicas da infraestrutura | Candidatas; presença não mede qualidade do uso |
| Horas-aula, biblioteca, água e outras condições | Alternativas e diagnóstico | Conservar na base ampla; revisar redundância antes do treinamento |
| N, cobertura e respostas válidas | Qualidade e composição | Guardar para auditoria; não usar como “nota de qualidade” |
| FUNDEB de dezembro | Contexto financeiro parcial | Excluir da lista principal até obter série anual de execução adequada |
| Perfis feitos com as notas do mesmo ano | Análise descritiva individual | Não usar para prever as próprias notas que os definiram |

O gráfico seguinte verifica redundância entre candidatos e jornada, usando uma linha por UF no ano escolhido. Não repetimos cada UF cinco vezes por causa das áreas. Correlação alta é um sinal para avaliar alternativas; não elimina automaticamente uma variável nem determina importância.

Esta lista é uma hipótese de trabalho. Nas próximas seções confrontamos a justificativa qualitativa com associação, redundância, influência de UFs e ganho em previsões temporais. Correlação de duas variáveis não basta para escolher entradas nem uma família de modelos.

In [ ]:

CANDIDATOS = ['RENDA_PC_REAIS_2024_IPCA', 'PCT_MATRICULAS_INTEGRAL_EM',
              'PCT_DOCENTES_ADEQUADOS_EM', 'PCT_ABANDONO_EM',
              'PCT_IN_INTERNET_ALUNOS', 'PCT_IN_LABORATORIO_CIENCIAS']
NOMES_ATRIBUTOS = {coluna: nome for nome, coluna in ATRIBUTOS.items()}

def mostrar_redundancia(ano, universo):
    origem = base_integrada if universo == 'Todas as redes' else base_estadual
    d = origem[origem.ANO.eq(ano)].drop_duplicates('UF_ESCOLA')
    colunas = CANDIDATOS + ['HORAS_AULA_DIA_EM']
    m = d[colunas].corr(method='spearman')
    nomes = ['Renda', 'Tempo integral', 'Formação docente', 'Abandono', 'Internet', 'Lab. ciências', 'Horas-aula']
    fig, ax = plt.subplots(figsize=(9, 6))
    im = ax.imshow(m, cmap='RdBu', vmin=-1, vmax=1)
    ax.set_xticks(range(len(m)), nomes, rotation=35, ha='right')
    ax.set_yticks(range(len(m)), nomes)
    for i in range(len(m)):
        for j in range(len(m)):
            v = m.iloc[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                    color='white' if abs(v) > .65 else '#243747')
    ax.set_title(f'Quais atributos podem trazer informações semelhantes para o modelo?· {ano} · {universo}')
    fig.colorbar(im, ax=ax, label='Spearman', shrink=.8)
    terminar(fig, '16_redundancia')

painel(mostrar_redundancia, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
    universo=escolher(['Todas as redes', 'Rede estadual'], 'Rede estadual', 'Universo:'))







## 44. Definir o que queremos prever e reconhecer o tamanho do problema

**Pergunta de continuidade:** com informações anteriores, conseguimos estimar a mediana do ENEM dos participantes da rede estadual em cada UF e área no ano seguinte? A utilidade seria sinalizar territórios para investigação de necessidades, não distribuir recursos automaticamente nem estimar o efeito de uma política.

Mantemos as 27 UFs porque são comparáveis nas três fontes durante 2020–2024. Selecionar somente São Paulo deixaria cinco registros anuais por área. Mudar para municípios ou escolas exigiria outra população-alvo, outras chaves e auditoria de cobertura; não produz mais informação comparável apenas por aumentar o número de linhas. O código de escola retornou ao ENEM público em 2024, conforme o Leia-me: não reconstruímos a série escolar anterior por notas ou posição de linha.

O gargalo é **tempo observado**, não volume de participantes: há 135 registros por área, 27 UFs repetidas por cinco anos. Os milhões de notas tornam os resumos mais precisos para o recorte observado, mas não criam milhões de exemplos territoriais para previsão.

Faremos ajustes simples de diagnóstico para testar a viabilidade da base. Eles não constituem o modelo final nem uma escolha já validada para uso público. Toda comparação de previsão usará anos posteriores ao treino. Como já exploramos 2020–2024, nenhum desses anos será chamado de teste cego.

A [página oficial do ENEM](https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/enem) já lista 2025 na consulta de 27/10/2026. Essa edição não foi incorporada nem avaliada aqui: poderá servir de validação adicional após congelar o protocolo, se o grupo também não a tiver explorado. Ainda será necessário verificar sua comparabilidade.


In [ ]:
from scipy.stats import pearsonr, rankdata, false_discovery_control, probplot
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.linear_model import Ridge
import sklearn

CONTEXTO = ['RENDA_PC_REAIS_2024_IPCA', 'PCT_MATRICULAS_INTEGRAL_EM',
            'PCT_DOCENTES_ADEQUADOS_EM', 'PCT_IN_INTERNET_ALUNOS',
            'PCT_IN_LABORATORIO_CIENCIAS', 'PCT_ABANDONO_EM']
ROTULOS = ['Renda', 'Tempo integral', 'Formação docente', 'Internet', 'Lab. ciências', 'Abandono']
ROTULO = dict(zip(CONTEXTO, ROTULOS))
SEMENTE_TESTES = 42
print('27 UFs × 5 anos = 135 registros por área. São cinco problemas separados.')
print('Ajustes exploratórios para diagnóstico; modelo final ainda não definido.')


27 UFs × 5 anos = 135 registros por área. São cinco problemas separados.
Ajustes exploratórios para diagnóstico; modelo final ainda não definido.




## 46. Relações lineares, monotônicas e outras formas de associação

Spearman sozinho não responde a toda a pergunta. **Pearson** resume alinhamento em uma reta; **Spearman** compara ordenações, aceitando relações crescentes ou decrescentes curvas; **correlação de distâncias** pode detectar outras dependências, inclusive formas em U. Esta última não tem sinal e seu valor amostral positivo não comprova associação.

Calculamos as três medidas em cada ano e área. Para avaliar sensibilidade a uma UF, repetimos Spearman retirando uma UF por vez: o intervalo mínimo–máximo é influência, não intervalo de confiança. Não excluímos um estado porque altera o resultado.

Para Pearson e distância, usamos 999 permutações do resultado **dentro de cada região**, preservando a composição regional. Essa referência pergunta se há mais associação do que a obtida ao reorganizar as UFs dentro das regiões. Ela pressupõe que essas trocas sejam razoáveis sob a hipótese nula; dependência espacial e diferenças internas limitam essa interpretação. Aplicamos correção Benjamini–Yekutieli sobre os 300 testes (6 atributos × 5 áreas × 5 anos × 2 medidas). Isso reduz falsas descobertas sob dependência entre testes, mas não corrige um teste cuja hipótese de permutação seja inadequada.

Um teste de distância significativo não prova que a relação seja especificamente não linear: relações lineares também são dependentes. A utilidade de permitir curvas será testada adiante, na previsão de anos posteriores. Fontes: [Pearson](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.pearsonr.html), [correlação de distâncias — artigo original](https://arxiv.org/abs/0803.4101), [correção de múltiplos testes](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.false_discovery_control.html).


In [ ]:
def matriz_distancias(x):
    distancias = abs(x[:, None] - x[None, :])
    return distancias - distancias.mean(axis=0) - distancias.mean(axis=1)[:, None] + distancias.mean()

def medir_associacoes(d, coluna, repeticoes=999):
    d = d[['UF_ESCOLA', 'REGIAO', coluna, 'MEDIANA']].dropna().reset_index(drop=True)
    x, y = d[coluna].to_numpy(float), d.MEDIANA.to_numpy(float)
    if len(x) < 4 or np.std(x) == 0 or np.std(y) == 0:
        return dict(N=len(x), PEARSON=np.nan, SPEARMAN=np.nan, DISTANCIA=np.nan,
                    P_PEARSON=np.nan, P_DISTANCIA=np.nan, RHO_MIN=np.nan, RHO_MAX=np.nan)
    pearson = pearsonr(x, y).statistic
    spearman = spearmanr(x, y).statistic
    a, b = matriz_distancias(x), matriz_distancias(y)
    denominador = np.sqrt(np.mean(a*a) * np.mean(b*b))
    produto = np.mean(a*b)
    distancia = np.sqrt(max(0, produto / denominador))
    rng = np.random.default_rng(SEMENTE_TESTES)
    regioes = [g.index.to_numpy() for _, g in d.groupby('REGIAO')]
    excede_p = excede_d = 0
    xc, yc = x - x.mean(), y - y.mean()
    norma = np.linalg.norm(xc) * np.linalg.norm(yc)
    for _ in range(repeticoes):
        ordem = np.arange(len(d))
        for indices in regioes:
            ordem[indices] = rng.permutation(indices)
        excede_p += abs(np.dot(xc, yc[ordem]) / norma) >= abs(pearson) - 1e-12
        excede_d += np.mean(a * b[np.ix_(ordem, ordem)]) >= produto - 1e-12
    influencia = [spearmanr(np.delete(x, i), np.delete(y, i)).statistic for i in range(len(d))]
    return dict(N=len(d), PEARSON=pearson, SPEARMAN=spearman, DISTANCIA=distancia,
        P_PEARSON=(excede_p+1)/(repeticoes+1), P_DISTANCIA=(excede_d+1)/(repeticoes+1),
        RHO_MIN=np.nanmin(influencia), RHO_MAX=np.nanmax(influencia))

associacoes_robustas = []
for (ano, area), d in base_estadual.groupby(['ANO', 'AREA'], observed=True):
    for coluna in CONTEXTO:
        associacoes_robustas.append(dict(ANO=ano, AREA=area, ATRIBUTO=coluna, **medir_associacoes(d, coluna)))
associacoes_robustas = pd.DataFrame(associacoes_robustas)
p = associacoes_robustas[['P_PEARSON', 'P_DISTANCIA']].to_numpy().ravel()
ajuste = np.full(len(p), np.nan)
ajuste[np.isfinite(p)] = false_discovery_control(p[np.isfinite(p)], method='by')
associacoes_robustas[['Q_PEARSON_BY', 'Q_DISTANCIA_BY']] = ajuste.reshape(-1, 2)
salvar(associacoes_robustas, 'associacoes_lineares_nao_lineares.csv')

def mostrar_formas_associacao(ano, area, atributo):
    d = base_estadual[base_estadual.ANO.eq(ano) & base_estadual.AREA.eq(area)]
    coluna = CONTEXTO[ROTULOS.index(atributo)]
    r = associacoes_robustas.query('ANO == @ano and AREA == @area and ATRIBUTO == @coluna').iloc[0]
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
    for regiao, g in d.groupby('REGIAO', observed=True):
        axs[0].scatter(g[coluna], g.MEDIANA, color=CORES_REGIOES[regiao], label=regiao, s=35)
    z = (d[coluna] - d[coluna].mean()) / d[coluna].std()
    eixo = np.linspace(z.min(), z.max(), 150)
    for grau, estilo in [(1, '-'), (2, '--')]:
        coef = np.polyfit(z, d.MEDIANA, grau)
        axs[0].plot(eixo*d[coluna].std()+d[coluna].mean(), np.polyval(coef, eixo), estilo,
                    color='#243747', label='Reta descritiva' if grau == 1 else 'Curva descritiva')
    axs[0].set(xlabel=atributo, ylabel='Mediana completa (pontos)', title=f'{ano} · rede estadual')
    axs[0].legend(frameon=False, fontsize=8)
    barras = axs[1].barh(['Pearson', 'Spearman', 'Distâncias'], [r.PEARSON, r.SPEARMAN, r.DISTANCIA],
                       color=['#2276A5', '#298574', '#C95C42'])
    axs[1].bar_label(barras, fmt='%.2f', padding=3)
    axs[1].set(xlim=(-1.1, 1.1), title='Medidas com significados diferentes')
    axs[1].axvline(0, color='#9CA9B1', linewidth=1)
    fig.suptitle(area); terminar(fig, '19_associacao_formas')
    print(f'Spearman retirando uma UF: {r.RHO_MIN:.2f} a {r.RHO_MAX:.2f} (não é intervalo de confiança).')
    print(f'p ajustado BY: Pearson={r.Q_PEARSON_BY:.3f}; distâncias={r.Q_DISTANCIA_BY:.3f}.')
    print('As curvas descrevem os pontos atuais. Ainda não demonstram capacidade de previsão.')

painel(mostrar_formas_associacao, ano=escolher(ANOS_ESTUDO, 2024, 'Ano:'),
       area=escolher(AREAS_ESTUDO, 'Matemática', 'Área:'), atributo=escolher(ROTULOS, 'Renda', 'Atributo:'))
